# Hybrid TCN Autoencoder–Binary Anomaly–TCN Classifier

Same complete 3-head workflow as the CNN, MLP, and LSTM versions, but temporal processing is performed with causal dilated Conv1D residual TCN blocks. The three outputs are reconstruction, binary anomaly probability, and seven-class classification.

Reconstruction is trained on the broader safe set (normal + motor_off + turning_on + turning_off), while the binary and multiclass heads retain the same labels, session-wise validation, conditional thresholds, fusion logic, and firmware exports.

In [ ]:
# ==========================================================
# CELL 0 — install dependencies
# ==========================================================

%pip install pandas numpy scikit-learn joblib openpyxl matplotlib tensorflow tf2onnx onnx onnxruntime

In [ ]:
# ==========================================================
# CELL 1 — imports
# ==========================================================
import os
import glob
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    StratifiedGroupKFold,
    StratifiedShuffleSplit,
)

from sklearn.utils.class_weight import compute_class_weight

In [ ]:
# ==========================================================
# CELL 2 — configuration
# ==========================================================
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

BASE_PATH = r"../../data/Motor_data_model"
SCALE_DIVISOR = 1_000_000.0
FIRMWARE_INPUT_DIVISOR = 1.0

FEATURE_COLUMNS = [
    "T_slope_10s", "PWM", "Irms", "Vrms", "Current_Variance",
    "Current_Crest_Factor", "Vib_RMS", "Vib_Min", "Vib_Q1",
    "Vib_Median", "Vib_Q3", "Vib_Max", "Vib_Variance", "Vib_Skewness",
]

FINAL_LABELS = [
    "motor_off", "turning_off", "turning_on", "normal",
    "electrical_fault", "mechanical_fault", "thermal_fault",
]

LABEL_TO_ID = {label: index for index, label in enumerate(FINAL_LABELS)}
SAFE_LABELS = ["motor_off", "turning_off", "turning_on", "normal"]
FAULT_LABELS = ["electrical_fault", "mechanical_fault", "thermal_fault"]

SAFE_CLASS_IDS = np.asarray([LABEL_TO_ID[l] for l in SAFE_LABELS], dtype=np.int64)
FAULT_CLASS_IDS = np.asarray([LABEL_TO_ID[l] for l in FAULT_LABELS], dtype=np.int64)

N_FEATURES = len(FEATURE_COLUMNS)
N_CLASSES = len(FINAL_LABELS)

WINDOW_LEN = 30
WINDOW_STEP = 10
ROW_PERIOD_SEC = 0.015

N_SPLITS = 5
BATCH_SIZE = 128
MAX_EPOCHS = 180
LEARNING_RATE = 1e-3

# CHANGED: RECON_LOSS_WEIGHT raised from 0.20 -> 1.00. Now that the
# reconstruction encoder is fully isolated (CELL 8 dual-encoder fix),
# it no longer competes with anomaly/class gradients for a shared
# TCN trunk, so a low weight is no longer needed to protect those
# heads — it was only slowing the recon branch's convergence.
RECON_LOSS_WEIGHT = 1.00
ANOMALY_LOSS_WEIGHT = 0.80
CLASS_LOSS_WEIGHT = 1.00

ENCODER_FILTERS_1 = 12
ENCODER_FILTERS_2 = 8
LATENT_FILTERS = 6
CLASS_DENSE_UNITS = 12
ANOMALY_DENSE_UNITS = 8

CLASS_CONF_THRESHOLD = 0.65
ANOMALY_PROB_THRESHOLD = 0.65
RECON_QUANTILE = 0.995

# ADDED: reconstruction severity tiers (matches DANN/CNN/LSTM notebooks).
SEVERITY_MILD_MULTIPLIER = 2.0
SEVERITY_MODERATE_MULTIPLIER = 4.0
SEVERITY_SEVERE_MULTIPLIER = 8.0

MOTOR_OFF_PWM_LIMIT = 0.01
MOTOR_OFF_IRMS_LIMIT = 0.01
MOTOR_OFF_VRMS_LIMIT = 0.01

# NOTE: the 1.01 upper edge already acts as a soft ceiling for PWM
# values slightly above 100% caused by ulong-scaling logging
# artifacts at full duty cycle. The real fix is clipping PWM at the
# source (see CELL 4) so those rows don't silently contaminate the
# "75_100" bin's calibration.
PWM_BIN_EDGES = np.asarray([
    0.00, 0.20, 0.35, 0.45, 0.55, 0.65, 0.75, 1.01,
], dtype=np.float32)

PWM_BIN_NAMES = ["0_20", "20_35", "35_45", "45_55", "55_65", "65_75", "75_100"]

LOAD_PROXY_FEATURE = "Irms"
LOAD_SPLIT_QUANTILE = 0.50
MIN_THRESHOLD_WINDOWS = 30
LOAD_STATE_NAMES = ["low_load", "high_load"]

OUTPUT_DIR = "pure_tcn_ae_classifier_3head_output"
CV_DIR = os.path.join(OUTPUT_DIR, "cross_validation")
FINAL_DIR = os.path.join(OUTPUT_DIR, "final_model")
MANUAL_TEST_DIR = os.path.join(OUTPUT_DIR, "manual_test_results")

for folder in [CV_DIR, FINAL_DIR, MANUAL_TEST_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Model: Pure 3-Head TCN (dual-encoder)")
print("Input:", WINDOW_LEN, "x", N_FEATURES)
print("Classes:", FINAL_LABELS)
print("Safe classes:", SAFE_LABELS)
print("Fault classes:", FAULT_LABELS)

In [ ]:
# ==========================================================
# CELL 3 — deterministic file-to-class mapping
# ==========================================================
def normalized_path(path):
    return str(path).replace("\\", "/").lower()


def label_for(path):
    path_text = normalized_path(path)
    stem = Path(path).stem.lower()

    if any(term in path_text for term in [
        "motor_being_off",
        "turning_off",
        "transition_off",
        "off_spikes",
    ]):
        return "turning_off"

    if any(term in path_text for term in [
        "motor_being_on",
        "turning_on",
        "transition_on",
        "on_spikes",
    ]):
        return "turning_on"

    if "/motor_off/" in path_text or "motor_off" in stem:
        return "motor_off"

    if any(term in path_text for term in [
        "electrical fault",
        "electrical_fault",
        "abnormal_current",
    ]):
        return "electrical_fault"

    if any(term in path_text for term in [
        "mechanical fault",
        "mechanical_fault",
        "abnormal_vibration",
    ]):
        return "mechanical_fault"

    if any(term in path_text for term in [
        "thermal fault",
        "thermal_fault",
        "abnormal_temp",
    ]):
        return "thermal_fault"

    if (
        "/normal/" in path_text
        or stem.startswith("normal_")
        or stem == "normal"
    ):
        return "normal"

    return None


def anomaly_target_from_class_id(class_id):
    return int(
        class_id in FAULT_CLASS_IDS
    )

# ----------------------------------------------------------
# CHANGED: reconstruction reference = NORMAL WITH LOAD only
# (same rule as the DANN notebook). Without-load normal,
# motor_off and transitions are NOT used to train the AE.
# ----------------------------------------------------------
def is_normal_with_load_file(path, label):
    if label != "normal":
        return False

    path_text = normalized_path(path)

    # Check without-load first because it also contains the word "load".
    without_load_terms = [
        "without_load",
        "without load",
        "no_load",
        "no load",
        "unloaded",
        "normal_noload",
        "normal_no_load",
    ]

    with_load_terms = [
        "with_load",
        "with load",
        "loaded",
        "normal_load",
    ]

    if any(term in path_text for term in without_load_terms):
        return False

    if any(term in path_text for term in with_load_terms):
        return True

    raise ValueError(
        "Cannot identify loaded/unloaded normal file: "
        f"{path}"
    )


In [ ]:
# ==========================================================
# CELL 4 — discover and read files
# ==========================================================
def discover_files(base_path):
    files = glob.glob(os.path.join(base_path, "**", "*.xlsx"), recursive=True)
    files += glob.glob(os.path.join(base_path, "**", "*.csv"), recursive=True)
    return sorted(set(files))


def read_one(path):
    path = Path(path)

    if path.suffix.lower() == ".xlsx":
        df = pd.read_excel(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    missing_columns = [c for c in FEATURE_COLUMNS if c not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing columns in {path.name}: {missing_columns}")

    clean = df[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce")
    clean = clean / SCALE_DIVISOR

    # ADDED: PWM duty cycle is physically bounded to [0, 1]. Values
    # recorded slightly above 1.0 are ulong-scaling logging artifacts
    # at the 100% boundary, not a real operating condition. Without
    # this clip they silently contaminate the top PWM bin's
    # reconstruction-threshold calibration.
    if "PWM" in clean.columns:
        overshoot_count = int((clean["PWM"] > 1.0).sum())
        if overshoot_count > 0:
            print(f"  Clipping {overshoot_count} PWM rows > 1.0 in {path.name}")
        clean["PWM"] = clean["PWM"].clip(upper=1.0)

    clean = clean.replace([np.inf, -np.inf], np.nan)
    clean = clean.dropna().reset_index(drop=True)

    if len(clean) < WINDOW_LEN:
        raise ValueError(f"{path.name} has only {len(clean)} valid rows.")

    return clean

In [ ]:
# ==========================================================
# CELL 5 — load labeled sessions
# ==========================================================
files = discover_files(BASE_PATH)

if not files:
    raise RuntimeError(
        f"No CSV or Excel files found under: {BASE_PATH}"
    )

sessions = []

for session_id, file_path in enumerate(files):
    label = label_for(file_path)

    if label is None:
        print(
            "SKIP unrecognized:",
            os.path.relpath(file_path, BASE_PATH),
        )
        continue

    try:
        df = read_one(file_path)
    except Exception as error:
        print(
            "SKIP read error:",
            os.path.relpath(file_path, BASE_PATH),
            "|",
            error,
        )
        continue

    class_id = LABEL_TO_ID[label]

    sessions.append({
        "session_id": session_id,
        "X": df.to_numpy(dtype=np.float32),
        "file": os.path.relpath(file_path, BASE_PATH),
        "label": label,
        "class_id": class_id,
        "anomaly": anomaly_target_from_class_id(class_id),
        "reconstruction_reference": int(is_normal_with_load_file(file_path, label)),  # CHANGED
        "rows": len(df),
    })

if not sessions:
    raise RuntimeError(
        "No labeled sessions were loaded."
    )

class_counts = pd.Series([
    session["label"]
    for session in sessions
]).value_counts().reindex(
    FINAL_LABELS,
    fill_value=0,
)

print("\nFILES PER CLASS")
print(class_counts)

missing_classes = [
    label
    for label in FINAL_LABELS
    if class_counts[label] == 0
]

if missing_classes:
    raise RuntimeError(
        f"No files found for classes: {missing_classes}"
    )


# ----------------------------------------------------------
# CHANGED: session_id -> 1 if normal-with-load (AE reference)
# ----------------------------------------------------------
RECON_REFERENCE_BY_SESSION = {
    session["session_id"]: session["reconstruction_reference"]
    for session in sessions
}

print("\nNORMAL WITH LOAD — USED TO TRAIN THE RECONSTRUCTION AE")
for session in sessions:
    if session["reconstruction_reference"] == 1:
        print("  ", session["file"])

print("\nNORMAL WITHOUT LOAD — NOT USED FOR AE TRAINING")
for session in sessions:
    if session["label"] == "normal" and session["reconstruction_reference"] == 0:
        print("  ", session["file"])

if not any(RECON_REFERENCE_BY_SESSION.values()):
    raise RuntimeError("No normal-with-load files were identified.")


In [ ]:
# ==========================================================
# CELL 6 — create 30 × 14 windows
# ==========================================================
def make_windows_from_array(X_raw):
    starts = np.arange(
        0,
        len(X_raw) - WINDOW_LEN + 1,
        WINDOW_STEP,
        dtype=np.int64,
    )

    if len(starts) == 0:
        return None, None

    windows = np.stack([
        X_raw[start:start + WINDOW_LEN]
        for start in starts
    ]).astype(np.float32)

    return windows, starts


def build_dataset_from_sessions(selected_sessions):
    all_windows = []
    all_class_labels = []
    all_anomaly_labels = []
    all_files = []
    all_starts = []
    all_groups = []

    for session in selected_sessions:
        windows, starts = make_windows_from_array(
            session["X"]
        )

        if windows is None:
            continue

        number_of_windows = len(windows)

        all_windows.append(windows)

        all_class_labels.extend(
            [session["class_id"]] * number_of_windows
        )

        all_anomaly_labels.extend(
            [session["anomaly"]] * number_of_windows
        )

        all_files.extend(
            [session["file"]] * number_of_windows
        )

        all_starts.extend(
            starts.tolist()
        )

        all_groups.extend(
            [session["session_id"]] * number_of_windows
        )

    if not all_windows:
        raise RuntimeError(
            "No windows were created."
        )

    return (
        np.concatenate(
            all_windows,
            axis=0,
        ).astype(np.float32),
        np.asarray(
            all_class_labels,
            dtype=np.int64,
        ),
        np.asarray(
            all_anomaly_labels,
            dtype=np.float32,
        ),
        np.asarray(
            all_files,
            dtype=object,
        ),
        np.asarray(
            all_starts,
            dtype=np.int64,
        ),
        np.asarray(
            all_groups,
            dtype=np.int64,
        ),
    )


In [ ]:
# ==========================================================
# CELL 7 — normalization utilities
# ==========================================================
def calculate_normalization_statistics(X_train):
    flattened_rows = X_train.reshape(
        -1,
        N_FEATURES,
    )

    mean = np.mean(
        flattened_rows,
        axis=0,
    ).astype(np.float32)

    variance = np.var(
        flattened_rows,
        axis=0,
    ).astype(np.float32)

    variance = np.maximum(
        variance,
        1e-12,
    )

    return mean, variance


def normalize_windows_numpy(
    X,
    mean,
    variance,
):
    return (
        (
            X
            - mean.reshape(
                1,
                1,
                N_FEATURES,
            )
        )
        / np.sqrt(
            variance.reshape(
                1,
                1,
                N_FEATURES,
            )
            + 1e-7
        )
    ).astype(np.float32)


In [ ]:
# ==========================================================
# CELL 8 — Cube.AI-compatible pure causal 3-head TCN
# DUAL-ENCODER VERSION.
#
# WHY THIS CHANGE:
# Previously `latent` (encoder_tcn_k3 -> encoder_tcn_k5 ->
# latent_tcn_k9) fed BOTH the reconstruction decoder AND the
# anomaly/class heads. anomaly_output/class_output train with full
# weight on every window, while reconstruction trains only on safe
# windows — so the classifier pulls the shared latent toward
# fault-discriminative structure, which then gets decoded,
# corrupting reconstruction error as an off-manifold signal. Same
# fix already applied to the DANN, CNN, and LSTM notebooks, so all
# four are now structurally aligned for a fair comparison.
#
# FIX: two fully independent causal-TCN encoder stacks, same three
# named outputs. All layer names for the reconstruction path are
# prefixed `recon_`; the discriminative path keeps the original
# names so downstream code (thresholds, fusion, manual test) does
# not need to change.
# ==========================================================
def window_mse_loss(y_true, y_pred):
    squared_error = tf.square(y_true - y_pred)
    return tf.reduce_mean(squared_error, axis=[1, 2])


def cube_causal_conv1d(inputs, filters, kernel_size, activation, layer_name):
    left_padding = kernel_size - 1

    x = tf.keras.layers.ZeroPadding1D(
        padding=(left_padding, 0), name=f"{layer_name}_left_padding",
    )(inputs)

    x = tf.keras.layers.Conv1D(
        filters=filters, kernel_size=kernel_size, strides=1,
        padding="valid", dilation_rate=1, activation=activation,
        use_bias=True, name=layer_name,
    )(x)

    return x


def cube_tcn_residual_block(inputs, filters, kernel_size, block_name):
    residual = inputs

    x = cube_causal_conv1d(
        inputs=inputs, filters=filters, kernel_size=kernel_size,
        activation=None, layer_name=f"{block_name}_conv1",
    )
    x = tf.keras.layers.ReLU(name=f"{block_name}_relu1")(x)

    x = cube_causal_conv1d(
        inputs=x, filters=filters, kernel_size=kernel_size,
        activation=None, layer_name=f"{block_name}_conv2",
    )

    if int(inputs.shape[-1]) != int(filters):
        residual = tf.keras.layers.Conv1D(
            filters=filters, kernel_size=1, strides=1, padding="same",
            dilation_rate=1, activation=None, use_bias=True,
            name=f"{block_name}_projection",
        )(residual)

    x = tf.keras.layers.Add(name=f"{block_name}_residual_add")([x, residual])
    x = tf.keras.layers.ReLU(name=f"{block_name}_output_relu")(x)

    return x


def build_pure_tcn_model(normalization_mean, normalization_variance):
    input_layer = tf.keras.Input(
        shape=(WINDOW_LEN, N_FEATURES), dtype=tf.float32, name="motor_window",
    )

    normalizer = tf.keras.layers.Normalization(
        axis=-1,
        mean=np.asarray(normalization_mean, dtype=np.float32),
        variance=np.asarray(normalization_variance, dtype=np.float32),
        name="input_normalization",
    )
    normalized = normalizer(input_layer)

    # ------------------------------------------------------
    # BRANCH A — dedicated reconstruction TCN encoder + decoder
    # (isolated). Only reconstruction_output's loss ever touches
    # these layers.
    # ------------------------------------------------------
    recon_encoded = cube_tcn_residual_block(
        inputs=normalized, filters=ENCODER_FILTERS_1,
        kernel_size=3, block_name="recon_encoder_tcn_k3",
    )
    recon_encoded = cube_tcn_residual_block(
        inputs=recon_encoded, filters=ENCODER_FILTERS_2,
        kernel_size=5, block_name="recon_encoder_tcn_k5",
    )
    recon_latent = cube_tcn_residual_block(
        inputs=recon_encoded, filters=LATENT_FILTERS,
        kernel_size=9, block_name="recon_latent_tcn_k9",
    )

    decoded = cube_tcn_residual_block(
        inputs=recon_latent, filters=ENCODER_FILTERS_2,
        kernel_size=9, block_name="decoder_tcn_k9",
    )
    decoded = cube_tcn_residual_block(
        inputs=decoded, filters=ENCODER_FILTERS_1,
        kernel_size=5, block_name="decoder_tcn_k5",
    )
    decoded = cube_tcn_residual_block(
        inputs=decoded, filters=ENCODER_FILTERS_1,
        kernel_size=3, block_name="decoder_tcn_k3",
    )

    RECON_OUTPUT_SCALE = 6.0

    reconstruction_bounded = tf.keras.layers.Conv1D(
        filters=N_FEATURES, kernel_size=1, strides=1, padding="same",
        dilation_rate=1, activation="tanh", use_bias=True,
        name="reconstruction_bounded",
    )(decoded)

    reconstruction_output = tf.keras.layers.Rescaling(
        scale=RECON_OUTPUT_SCALE, name="reconstruction_output",
    )(reconstruction_bounded)

    # ------------------------------------------------------
    # BRANCH B — shared discriminative TCN encoder (anomaly +
    # class only). Completely separate weights from Branch A.
    # ------------------------------------------------------
    encoded = cube_tcn_residual_block(
        inputs=normalized, filters=ENCODER_FILTERS_1,
        kernel_size=3, block_name="encoder_tcn_k3",
    )
    encoded = cube_tcn_residual_block(
        inputs=encoded, filters=ENCODER_FILTERS_2,
        kernel_size=5, block_name="encoder_tcn_k5",
    )
    latent = cube_tcn_residual_block(
        inputs=encoded, filters=LATENT_FILTERS,
        kernel_size=9, block_name="latent_tcn_k9",
    )

    anomaly_sequence = cube_tcn_residual_block(
        inputs=latent, filters=ANOMALY_DENSE_UNITS,
        kernel_size=5, block_name="anomaly_tcn_k5",
    )
    anomaly_sequence = cube_tcn_residual_block(
        inputs=anomaly_sequence, filters=ANOMALY_DENSE_UNITS,
        kernel_size=9, block_name="anomaly_tcn_k9",
    )
    anomaly_features = tf.keras.layers.GlobalAveragePooling1D(
        name="anomaly_global_average_pool",
    )(anomaly_sequence)
    anomaly_output = tf.keras.layers.Dense(
        units=1, activation="sigmoid", use_bias=True, name="anomaly_output",
    )(anomaly_features)

    class_sequence = cube_tcn_residual_block(
        inputs=latent, filters=CLASS_DENSE_UNITS,
        kernel_size=5, block_name="class_tcn_k5",
    )
    class_sequence = cube_tcn_residual_block(
        inputs=class_sequence, filters=CLASS_DENSE_UNITS,
        kernel_size=9, block_name="class_tcn_k9",
    )
    class_features = tf.keras.layers.GlobalAveragePooling1D(
        name="class_global_average_pool",
    )(class_sequence)
    class_output = tf.keras.layers.Dense(
        units=N_CLASSES, activation="softmax", use_bias=True, name="class_output",
    )(class_features)

    model = tf.keras.Model(
        inputs=input_layer,
        outputs={
            "reconstruction_output": reconstruction_output,
            "anomaly_output": anomaly_output,
            "class_output": class_output,
        },
        name="dual_encoder_pure_cube_compatible_tcn_3head",
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss={
            "reconstruction_output": window_mse_loss,
            "anomaly_output": tf.keras.losses.BinaryCrossentropy(),
            "class_output": tf.keras.losses.SparseCategoricalCrossentropy(),
        },
        loss_weights={
            "reconstruction_output": RECON_LOSS_WEIGHT,
            "anomaly_output": ANOMALY_LOSS_WEIGHT,
            "class_output": CLASS_LOSS_WEIGHT,
        },
        metrics={
            "anomaly_output": [
                tf.keras.metrics.BinaryAccuracy(name="accuracy"),
                tf.keras.metrics.AUC(name="auc"),
            ],
            "class_output": [
                tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
            ],
        },
    )

    return model


preview_model = build_pure_tcn_model(
    normalization_mean=np.zeros(N_FEATURES, dtype=np.float32),
    normalization_variance=np.ones(N_FEATURES, dtype=np.float32),
)

preview_model.summary()

preview_parameters = preview_model.count_params()
preview_float32_kb = preview_parameters * 4 / 1024
preview_int8_weight_kb = preview_parameters / 1024

print("\nMODEL SIZE ESTIMATE")
print("Total parameters:", preview_parameters)
print("Estimated float32 weight size KB:", round(preview_float32_kb, 2))
print("Estimated INT8 weight size KB:", round(preview_int8_weight_kb, 2))

print("\nCube.AI compatibility changes:")
print("- All Conv1D layers use dilation_rate=1")
print("- Causality uses fixed left padding")
print("- No SPACE_TO_BATCH_ND dilation implementation")
print("- Dual-encoder: reconstruction path fully isolated from anomaly/class")
print("- Fixed input shape:", (1, WINDOW_LEN, N_FEATURES))
print("- Three outputs: reconstruction, anomaly and class")

del preview_model
tf.keras.backend.clear_session()

In [ ]:
# ==========================================================
# CELL 9 — training weights and dataset split helpers
# ==========================================================
def calculate_class_sample_weights(y_class):
    present_classes = np.unique(y_class)

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=present_classes,
        y=y_class,
    )

    class_weight_lookup = {
        int(class_id): float(weight)
        for class_id, weight in zip(
            present_classes,
            class_weights,
        )
    }

    return np.asarray([
        class_weight_lookup[int(class_id)]
        for class_id in y_class
    ], dtype=np.float32)


def calculate_anomaly_sample_weights(y_anomaly):
    y_integer = y_anomaly.astype(np.int64)
    present_classes = np.unique(y_integer)

    weights = compute_class_weight(
        class_weight="balanced",
        classes=present_classes,
        y=y_integer,
    )

    lookup = {
        int(class_id): float(weight)
        for class_id, weight in zip(
            present_classes,
            weights,
        )
    }

    return np.asarray([
        lookup[int(value)]
        for value in y_integer
    ], dtype=np.float32)


def recon_reference_for_groups(groups):
    """CHANGED: 1.0 for windows from normal-with-load files, else 0.0."""
    return np.asarray(
        [RECON_REFERENCE_BY_SESSION[int(g)] for g in groups],
        dtype=np.float32,
    )


def make_training_sample_weights(
    y_class,
    y_anomaly,
    groups,
):
    # CHANGED: AE trains ONLY on normal-with-load windows
    reconstruction_weights = recon_reference_for_groups(groups)

    class_weights = calculate_class_sample_weights(
        y_class
    )

    anomaly_weights = calculate_anomaly_sample_weights(
        y_anomaly
    )

    return {
        "reconstruction_output":
            reconstruction_weights,
        "anomaly_output":
            anomaly_weights,
        "class_output":
            class_weights,
    }


def split_train_validation_by_group(
    X,
    y_class,
    y_anomaly,
    files,
    starts,
    groups,
):
    """
    Create validation data at SESSION level, never by overlapping windows.
    This avoids leakage and does not require StratifiedGroupShuffleSplit,
    which is unavailable in several scikit-learn versions.
    """
    unique_groups, first_indices = np.unique(
        groups,
        return_index=True,
    )
    group_labels = y_class[first_indices]

    class_group_counts = pd.Series(
        group_labels
    ).value_counts()

    if class_group_counts.min() < 2:
        raise RuntimeError(
            "Each class needs at least 2 sessions inside the training fold "
            "to create a stratified session-wise validation split. "
            f"Current session counts: {class_group_counts.to_dict()}"
        )

    validation_fraction = max(
        0.15,
        N_CLASSES / len(unique_groups),
    )
    validation_fraction = min(
        validation_fraction,
        0.40,
    )

    splitter = StratifiedShuffleSplit(
        n_splits=1,
        test_size=validation_fraction,
        random_state=SEED,
    )

    group_train_pos, group_validation_pos = next(
        splitter.split(
            unique_groups,
            group_labels,
        )
    )

    train_group_ids = unique_groups[group_train_pos]
    validation_group_ids = unique_groups[group_validation_pos]

    train_index = np.flatnonzero(
        np.isin(groups, train_group_ids)
    )
    validation_index = np.flatnonzero(
        np.isin(groups, validation_group_ids)
    )

    if np.intersect1d(
        groups[train_index],
        groups[validation_index],
    ).size:
        raise RuntimeError(
            "Session leakage detected between training and validation."
        )

    train_data = (
        X[train_index],
        y_class[train_index],
        y_anomaly[train_index],
        files[train_index],
        starts[train_index],
        groups[train_index],
    )

    validation_data = (
        X[validation_index],
        y_class[validation_index],
        y_anomaly[validation_index],
        files[validation_index],
        starts[validation_index],
        groups[validation_index],
    )

    return train_data, validation_data

In [ ]:
# ==========================================================
# CELL 10 — fit one fold/model
# ==========================================================
def fit_model_with_group_validation(
    X_train_full,
    y_class_full,
    y_anomaly_full,
    files_full,
    starts_full,
    groups_full,
):
    (
        train_data,
        validation_data,
    ) = split_train_validation_by_group(
        X_train_full,
        y_class_full,
        y_anomaly_full,
        files_full,
        starts_full,
        groups_full,
    )

    (
        X_train,
        y_class_train,
        y_anomaly_train,
        _,
        _,
        groups_train,
    ) = train_data

    (
        X_validation,
        y_class_validation,
        y_anomaly_validation,
        _,
        _,
        groups_validation,
    ) = validation_data

    normalization_mean, normalization_variance = (
        calculate_normalization_statistics(
            X_train
        )
    )

    X_train_normalized = normalize_windows_numpy(
        X_train,
        normalization_mean,
        normalization_variance,
    )

    X_validation_normalized = normalize_windows_numpy(
        X_validation,
        normalization_mean,
        normalization_variance,
    )

    model = build_pure_tcn_model(
        normalization_mean,
        normalization_variance,
    )

    sample_weights = make_training_sample_weights(
        y_class_train,
        y_anomaly_train,
        groups_train,
    )

    validation_sample_weights = {
        "reconstruction_output":
            recon_reference_for_groups(groups_validation),  # CHANGED
        "anomaly_output":
            np.ones(
                len(y_anomaly_validation),
                dtype=np.float32,
            ),
        "class_output":
            np.ones(
                len(y_class_validation),
                dtype=np.float32,
            ),
    }

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_class_output_accuracy",
            mode="max",
            patience=20,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_class_output_loss",
            mode="min",
            factor=0.5,
            patience=7,
            min_lr=1e-5,
            verbose=1,
        ),
    ]

    history = model.fit(
        X_train,
        {
            "reconstruction_output":
                X_train_normalized,
            "anomaly_output":
                y_anomaly_train,
            "class_output":
                y_class_train,
        },
        sample_weight=sample_weights,
        validation_data=(
            X_validation,
            {
                "reconstruction_output":
                    X_validation_normalized,
                "anomaly_output":
                    y_anomaly_validation,
                "class_output":
                    y_class_validation,
            },
            validation_sample_weights,
        ),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=2,
        shuffle=True,
    )

    return (
        model,
        normalization_mean,
        normalization_variance,
        history,
    )


In [ ]:
# ==========================================================
# CELL 11 — decision fusion and conditional thresholds
# (CHANGED: added severity_level_and_action() and
# get_local_normal_baseline(). fuse_decisions now also returns
# severity_ratio/severity_label/recommended_action per window.)
# ==========================================================
def get_pwm_bin_index(mean_pwm):
    bin_index = np.searchsorted(PWM_BIN_EDGES, mean_pwm, side="right") - 1
    return int(np.clip(bin_index, 0, len(PWM_BIN_NAMES) - 1))


def get_local_normal_baseline(mean_pwm):
    """ADDED — per-bin median reconstruction MSE for severity scoring."""
    bin_index = get_pwm_bin_index(mean_pwm)
    bin_name = PWM_BIN_NAMES[bin_index]
    return float(LOCAL_NORMAL_BASELINES.get(bin_name, GLOBAL_NORMAL_BASELINE))


def severity_level_and_action(severity_ratio):
    """ADDED — matches DANN/CNN/LSTM severity tiering exactly."""
    if severity_ratio < SEVERITY_MILD_MULTIPLIER:
        return "normal", "No action needed."
    elif severity_ratio < SEVERITY_MODERATE_MULTIPLIER:
        return "mild", "Log and monitor; no immediate action."
    elif severity_ratio < SEVERITY_SEVERE_MULTIPLIER:
        return "moderate", "Schedule inspection soon."
    else:
        return "severe", "Stop motor / inspect immediately."


def get_conditional_reconstruction_threshold(
    predicted_class_id, class_confidence, mean_pwm, mean_irms, mean_vrms,
):
    

    
    predicted_label = FINAL_LABELS[int(predicted_class_id)]

    if (
        predicted_label in ["turning_off", "turning_on"]
        and class_confidence >= CLASS_CONF_THRESHOLD
    ):
        return float(STATE_RECON_THRESHOLDS[predicted_label])

    bin_index = get_pwm_bin_index(mean_pwm)
    bin_name = PWM_BIN_NAMES[bin_index]

    load_split = float(PWM_LOAD_SPLITS.get(bin_name, GLOBAL_NORMAL_LOAD_SPLIT))
    load_state = "high_load" if mean_irms >= load_split else "low_load"

    return float(
        NORMAL_PWM_LOAD_RECON_THRESHOLDS.get(bin_name, {}).get(
            load_state,
            NORMAL_PWM_RECON_THRESHOLDS.get(bin_name, RECON_THRESHOLD),
        )
    )


def fuse_decisions(
    class_probabilities, anomaly_probabilities, reconstruction_mse,
    reconstruction_thresholds, physical_motor_off_flags, mean_pwm,
):
    class_ids = np.argmax(class_probabilities, axis=1).astype(np.int64)
    class_confidence = np.max(class_probabilities, axis=1)

    reconstruction_anomaly_flags = reconstruction_mse > reconstruction_thresholds
    binary_anomaly_flags = anomaly_probabilities >= ANOMALY_PROB_THRESHOLD

    confident_known_fault_flags = (
        np.isin(class_ids, FAULT_CLASS_IDS)
        & (class_confidence >= CLASS_CONF_THRESHOLD)
    )

    physical_motor_off_flags = np.asarray(physical_motor_off_flags, dtype=bool)

    anomaly_flags = (
        binary_anomaly_flags | reconstruction_anomaly_flags | confident_known_fault_flags
    )

    # ADDED: severity ratio + tier, informational for every window.
    local_baselines = np.asarray(
        [get_local_normal_baseline(pwm) for pwm in mean_pwm], dtype=np.float32,
    )
    safe_baselines = np.where(local_baselines > 0, local_baselines, 1.0)
    severity_ratios = reconstruction_mse / safe_baselines

    severity_labels = []
    severity_actions = []
    for ratio in severity_ratios:
        label, action = severity_level_and_action(float(ratio))
        severity_labels.append(label)
        severity_actions.append(action)

    final_labels = []

    for (
        class_id, confidence, binary_flag, reconstruction_flag,
        known_fault_flag, physical_motor_off,
    ) in zip(
        class_ids, class_confidence, binary_anomaly_flags,
        reconstruction_anomaly_flags, confident_known_fault_flags,
        physical_motor_off_flags,
    ):
        predicted_label = FINAL_LABELS[int(class_id)]

       
        if known_fault_flag:
            final_labels.append(predicted_label)
        elif binary_flag or reconstruction_flag:
            final_labels.append("unknown_fault")
        elif class_id in SAFE_CLASS_IDS and confidence >= CLASS_CONF_THRESHOLD:
            final_labels.append(predicted_label)
        else:
            final_labels.append("uncertain_safe")

    return (
        class_ids,
        class_confidence,
        binary_anomaly_flags,
        reconstruction_anomaly_flags,
        anomaly_flags,
        np.asarray(final_labels, dtype=object),
        local_baselines,
        severity_ratios.astype(np.float32),
        np.asarray(severity_labels, dtype=object),
        np.asarray(severity_actions, dtype=object),
    )

In [ ]:
# ==========================================================
# CELL 12 — five-fold file-wise cross-validation
# ==========================================================
cv_table = pd.DataFrame({
    "label": [
        session["class_id"]
        for session in sessions
    ],
    "group": [
        session["session_id"]
        for session in sessions
    ],
})

splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

cv_metrics = []
all_true = []
all_pred = []
all_anomaly_true = []
all_anomaly_pred = []
all_reconstruction_mse = []
all_mean_pwm = []
all_mean_irms = []
all_window_predictions = []
cv_best_epochs = []

for fold, (
    train_index,
    test_index,
) in enumerate(
    splitter.split(
        cv_table,
        cv_table["label"],
        groups=cv_table["group"],
    ),
    start=1,
):
    print("\n" + "=" * 70)
    print("FOLD", fold)
    print("=" * 70)

    train_sessions = [
        sessions[index]
        for index in train_index
    ]

    test_sessions = [
        sessions[index]
        for index in test_index
    ]

    (
        X_train_full,
        y_class_train_full,
        y_anomaly_train_full,
        train_files,
        train_starts,
        train_groups,
    ) = build_dataset_from_sessions(
        train_sessions
    )

    (
        X_test,
        y_class_test,
        y_anomaly_test,
        test_files,
        test_starts,
        _,
    ) = build_dataset_from_sessions(
        test_sessions
    )

    (
        model,
        normalization_mean,
        normalization_variance,
        history,
    ) = fit_model_with_group_validation(
        X_train_full,
        y_class_train_full,
        y_anomaly_train_full,
        train_files,
        train_starts,
        train_groups,
    )

    validation_accuracy_history = history.history.get(
        "val_class_output_accuracy",
        [],
    )
    if validation_accuracy_history:
        best_epoch = int(
            np.argmax(validation_accuracy_history) + 1
        )
    else:
        best_epoch = len(history.history.get("loss", []))

    cv_best_epochs.append(best_epoch)
    print("Best validation epoch:", best_epoch)

    X_test_normalized = normalize_windows_numpy(
        X_test,
        normalization_mean,
        normalization_variance,
    )

    prediction_dict = model.predict(
        X_test,
        batch_size=BATCH_SIZE,
        verbose=0,
    )

    reconstructed = prediction_dict[
        "reconstruction_output"
    ]

    anomaly_probabilities = prediction_dict[
        "anomaly_output"
    ].reshape(-1)

    class_probabilities = prediction_dict[
        "class_output"
    ]

    class_predictions = np.argmax(
        class_probabilities,
        axis=1,
    ).astype(np.int64)

    anomaly_predictions = (
        anomaly_probabilities
        >= ANOMALY_PROB_THRESHOLD
    ).astype(np.int64)

    reconstruction_mse = np.mean(
        np.square(
            reconstructed
            - X_test_normalized
        ),
        axis=(1, 2),
    )

    pwm_feature_index = FEATURE_COLUMNS.index(
        "PWM"
    )
    irms_feature_index = FEATURE_COLUMNS.index(
        LOAD_PROXY_FEATURE
    )
    vrms_feature_index = FEATURE_COLUMNS.index(
        "Vrms"
    )

    test_window_mean_pwm = np.mean(
        X_test[:, :, pwm_feature_index],
        axis=1,
    ).astype(np.float32)

    test_window_mean_irms = np.mean(
        X_test[:, :, irms_feature_index],
        axis=1,
    ).astype(np.float32)
    

    all_true.extend(
        y_class_test.tolist()
    )

    all_pred.extend(
        class_predictions.tolist()
    )

    all_anomaly_true.extend(
        y_anomaly_test.astype(np.int64).tolist()
    )

    all_anomaly_pred.extend(
        anomaly_predictions.tolist()
    )

    all_reconstruction_mse.extend(
        reconstruction_mse.tolist()
    )

    all_mean_pwm.extend(
        test_window_mean_pwm.tolist()
    )

    all_mean_irms.extend(
        test_window_mean_irms.tolist()
    )

    fold_metrics = {
        "fold": fold,
        "accuracy":
            accuracy_score(
                y_class_test,
                class_predictions,
            ),
        "balanced_accuracy":
            balanced_accuracy_score(
                y_class_test,
                class_predictions,
            ),
        "macro_precision":
            precision_score(
                y_class_test,
                class_predictions,
                average="macro",
                zero_division=0,
            ),
        "macro_recall":
            recall_score(
                y_class_test,
                class_predictions,
                average="macro",
                zero_division=0,
            ),
        "macro_f1":
            f1_score(
                y_class_test,
                class_predictions,
                average="macro",
                zero_division=0,
            ),
        "anomaly_accuracy":
            accuracy_score(
                y_anomaly_test,
                anomaly_predictions,
            ),
        "anomaly_f1":
            f1_score(
                y_anomaly_test,
                anomaly_predictions,
                zero_division=0,
            ),
        "mean_reconstruction_mse":
            float(
                np.mean(
                    reconstruction_mse
                )
            ),
    }

    cv_metrics.append(
        fold_metrics
    )

    fold_predictions = pd.DataFrame({
        "fold": fold,
        "file": test_files,
        "start_row": test_starts,
        "mean_pwm": test_window_mean_pwm,
        "mean_irms": test_window_mean_irms,
        "true_class_id":
            y_class_test,
        "true_label": [
            FINAL_LABELS[index]
            for index in y_class_test
        ],
        "pred_class_id":
            class_predictions,
        "pred_label": [
            FINAL_LABELS[index]
            for index in class_predictions
        ],
        "class_confidence":
            np.max(
                class_probabilities,
                axis=1,
            ),
        "true_anomaly":
            y_anomaly_test.astype(np.int64),
        "anomaly_probability":
            anomaly_probabilities,
        "anomaly_prediction":
            anomaly_predictions,
        "reconstruction_mse":
            reconstruction_mse,
    })

    for class_id, label in enumerate(
        FINAL_LABELS
    ):
        fold_predictions[
            f"probability_{label}"
        ] = class_probabilities[
            :,
            class_id,
        ]

    all_window_predictions.append(
        fold_predictions
    )

    print(
        pd.Series(
            fold_metrics
        )
    )

    print("\nFOLD CLASSIFICATION REPORT")
    print(
        classification_report(
            y_class_test,
            class_predictions,
            labels=np.arange(N_CLASSES),
            target_names=FINAL_LABELS,
            zero_division=0,
        )
    )

    fold_cm = confusion_matrix(
        y_class_test,
        class_predictions,
        labels=np.arange(N_CLASSES),
    )

    pd.DataFrame(
        fold_cm,
        index=FINAL_LABELS,
        columns=FINAL_LABELS,
    ).to_csv(
        os.path.join(
            CV_DIR,
            f"fold_{fold}_confusion_matrix.csv",
        )
    )

    fig, axis = plt.subplots(figsize=(9, 8))
    ConfusionMatrixDisplay(
        fold_cm,
        display_labels=FINAL_LABELS,
    ).plot(
        ax=axis,
        values_format="d",
        xticks_rotation=45,
        colorbar=False,
    )
    axis.set_title(f"Fold {fold} — Session-wise Test Confusion Matrix")
    fig.tight_layout()
    fig.savefig(
        os.path.join(
            CV_DIR,
            f"fold_{fold}_confusion_matrix.png",
        ),
        dpi=180,
    )
    plt.show()
    plt.close(fig)

    history_df = pd.DataFrame(history.history)
    history_df.to_csv(
        os.path.join(
            CV_DIR,
            f"fold_{fold}_training_history.csv",
        ),
        index=False,
    )

    tf.keras.backend.clear_session()

In [ ]:
# ==========================================================
# CELL 13 — combined CV report and reconstruction threshold
# (CHANGED: added GLOBAL_NORMAL_BASELINE and LOCAL_NORMAL_BASELINES,
# needed by the severity scoring added in CELL 11.)
# ==========================================================
cv_metrics_df = pd.DataFrame(cv_metrics)
cv_predictions_df = pd.concat(all_window_predictions, ignore_index=True)

cv_metrics_df.to_csv(os.path.join(CV_DIR, "cv_metrics.csv"), index=False)
cv_predictions_df.to_csv(os.path.join(CV_DIR, "cv_window_predictions.csv"), index=False)

all_true = np.asarray(all_true, dtype=np.int64)
all_pred = np.asarray(all_pred, dtype=np.int64)
all_anomaly_true = np.asarray(all_anomaly_true, dtype=np.int64)
all_anomaly_pred = np.asarray(all_anomaly_pred, dtype=np.int64)
all_mean_pwm = np.asarray(all_mean_pwm, dtype=np.float32)
all_mean_irms = np.asarray(all_mean_irms, dtype=np.float32)
all_reconstruction_mse = np.asarray(all_reconstruction_mse, dtype=np.float32)

NORMAL_CLASS_ID = LABEL_TO_ID["normal"]

safe_mask = np.isin(all_true, SAFE_CLASS_IDS)

if not np.any(safe_mask):
    raise RuntimeError(
        "No held-out safe windows are available for reconstruction-threshold calibration."
    )

safe_reconstruction_mse = all_reconstruction_mse[safe_mask]
RECON_THRESHOLD = float(np.quantile(safe_reconstruction_mse, RECON_QUANTILE))

STATE_RECON_THRESHOLDS = {}

for state_label in ["motor_off", "turning_off", "turning_on"]:
    state_class_id = LABEL_TO_ID[state_label]
    state_errors = all_reconstruction_mse[all_true == state_class_id]

    if len(state_errors) >= MIN_THRESHOLD_WINDOWS:
        state_threshold = float(np.quantile(state_errors, RECON_QUANTILE))
    else:
        state_threshold = RECON_THRESHOLD
        print(
            f"WARNING: state {state_label} has only {len(state_errors)} "
            "held-out windows. Using global safe fallback threshold."
        )

    STATE_RECON_THRESHOLDS[state_label] = state_threshold
    print(
        f"State {state_label} | windows={len(state_errors)} | "
        f"threshold={state_threshold:.8f}"
    )

normal_mask = all_true == NORMAL_CLASS_ID

if not np.any(normal_mask):
    raise RuntimeError(
        "No held-out steady normal windows are available for PWM/load threshold calibration."
    )

normal_reconstruction_mse = all_reconstruction_mse[normal_mask]
normal_mean_pwm = all_mean_pwm[normal_mask]
normal_mean_irms = all_mean_irms[normal_mask]

GLOBAL_NORMAL_LOAD_SPLIT = float(np.quantile(normal_mean_irms, LOAD_SPLIT_QUANTILE))
GLOBAL_NORMAL_THRESHOLD = float(np.quantile(normal_reconstruction_mse, RECON_QUANTILE))

# ADDED: global baseline for severity scoring fallback.
GLOBAL_NORMAL_BASELINE = float(np.median(normal_reconstruction_mse))

PWM_LOAD_SPLITS = {}
NORMAL_PWM_RECON_THRESHOLDS = {}
NORMAL_PWM_LOAD_RECON_THRESHOLDS = {}
# ADDED
LOCAL_NORMAL_BASELINES = {}
threshold_rows = []

for bin_index, bin_name in enumerate(PWM_BIN_NAMES):
    lower_pwm = PWM_BIN_EDGES[bin_index]
    upper_pwm = PWM_BIN_EDGES[bin_index + 1]

    pwm_mask = (normal_mean_pwm >= lower_pwm) & (normal_mean_pwm < upper_pwm)
    pwm_errors = normal_reconstruction_mse[pwm_mask]
    pwm_irms = normal_mean_irms[pwm_mask]

    if len(pwm_errors) >= MIN_THRESHOLD_WINDOWS:
        pwm_threshold = float(np.quantile(pwm_errors, RECON_QUANTILE))
        pwm_baseline = float(np.median(pwm_errors))
        load_split = float(np.quantile(pwm_irms, LOAD_SPLIT_QUANTILE))
    else:
        pwm_threshold = GLOBAL_NORMAL_THRESHOLD
        pwm_baseline = GLOBAL_NORMAL_BASELINE
        load_split = GLOBAL_NORMAL_LOAD_SPLIT
        print(
            f"WARNING: PWM bin {bin_name} has only {len(pwm_errors)} "
            "normal windows. Using global normal fallback values."
        )

    PWM_LOAD_SPLITS[bin_name] = load_split
    NORMAL_PWM_RECON_THRESHOLDS[bin_name] = pwm_threshold
    LOCAL_NORMAL_BASELINES[bin_name] = pwm_baseline
    NORMAL_PWM_LOAD_RECON_THRESHOLDS[bin_name] = {}

    for load_state in LOAD_STATE_NAMES:
        if load_state == "low_load":
            load_mask = pwm_mask & (normal_mean_irms < load_split)
        else:
            load_mask = pwm_mask & (normal_mean_irms >= load_split)

        load_errors = normal_reconstruction_mse[load_mask]

        if len(load_errors) >= MIN_THRESHOLD_WINDOWS:
            load_threshold = float(np.quantile(load_errors, RECON_QUANTILE))
        else:
            load_threshold = pwm_threshold
            print(
                f"WARNING: {bin_name}/{load_state} has only {len(load_errors)} "
                "normal windows. Using its PWM-bin threshold."
            )

        NORMAL_PWM_LOAD_RECON_THRESHOLDS[bin_name][load_state] = load_threshold

        threshold_rows.append({
            "operating_state": "normal",
            "pwm_bin": bin_name,
            "lower_pwm": float(lower_pwm),
            "upper_pwm": float(upper_pwm),
            "load_state": load_state,
            "irms_split": float(load_split),
            "normal_windows": int(len(load_errors)),
            "reconstruction_threshold": float(load_threshold),
            "local_baseline": float(pwm_baseline),
        })

        print(
            f"Normal {bin_name}/{load_state} | windows={len(load_errors)} | "
            f"Irms split={load_split:.8f} | threshold={load_threshold:.8f}"
        )

threshold_table = pd.DataFrame(threshold_rows)
threshold_table.to_csv(
    os.path.join(CV_DIR, "conditional_reconstruction_thresholds.csv"), index=False,
)

print("\nGlobal safe reconstruction fallback:", RECON_THRESHOLD)

overall_metrics = {
    "accuracy": accuracy_score(all_true, all_pred),
    "balanced_accuracy": balanced_accuracy_score(all_true, all_pred),
    "macro_precision": precision_score(all_true, all_pred, average="macro", zero_division=0),
    "macro_recall": recall_score(all_true, all_pred, average="macro", zero_division=0),
    "macro_f1": f1_score(all_true, all_pred, average="macro", zero_division=0),
    "anomaly_accuracy": accuracy_score(all_anomaly_true, all_anomaly_pred),
    "anomaly_f1": f1_score(all_anomaly_true, all_anomaly_pred, zero_division=0),
    "reconstruction_threshold": RECON_THRESHOLD,
}

print("\nOVERALL CROSS-VALIDATION RESULTS")
print(pd.Series(overall_metrics))

print(
    classification_report(
        all_true, all_pred, labels=np.arange(N_CLASSES),
        target_names=FINAL_LABELS, zero_division=0,
    )
)

pd.DataFrame([overall_metrics]).to_csv(
    os.path.join(CV_DIR, "overall_metrics.csv"), index=False,
)

ConfusionMatrixDisplay(
    confusion_matrix(all_true, all_pred, labels=np.arange(N_CLASSES)),
    display_labels=FINAL_LABELS,
).plot(values_format="d", xticks_rotation=45)

plt.title("Pure 3-Head TCN Cross-Validation")
plt.tight_layout()
plt.savefig(os.path.join(CV_DIR, "confusion_matrix.png"), dpi=180)
plt.show()

In [ ]:
# ==========================================================
# CELL 14 — train final model on all sessions
# ==========================================================
(
    X_all,
    y_class_all,
    y_anomaly_all,
    all_files,
    all_starts,
    all_groups,
) = build_dataset_from_sessions(
    sessions
)

final_mean, final_variance = calculate_normalization_statistics(
    X_all
)

X_all_normalized = normalize_windows_numpy(
    X_all,
    final_mean,
    final_variance,
)

final_model = build_pure_tcn_model(
    final_mean,
    final_variance,
)

final_sample_weights = make_training_sample_weights(
    y_class_all,
    y_anomaly_all,
    all_groups,
)

FINAL_EPOCHS = int(
    np.median(cv_best_epochs)
)
FINAL_EPOCHS = max(FINAL_EPOCHS, 1)

print("\n" + "=" * 70)
print("FINAL DEPLOYMENT TRAINING ON 100% OF DATA")
print("=" * 70)
print("Sessions used:", len(np.unique(all_groups)))
print("Windows used:", len(X_all))
print("Final epochs from median CV best epoch:", FINAL_EPOCHS)
print("No validation split is removed from the final training dataset.")

final_history = final_model.fit(
    X_all,
    {
        "reconstruction_output": X_all_normalized,
        "anomaly_output": y_anomaly_all,
        "class_output": y_class_all,
    },
    sample_weight=final_sample_weights,
    epochs=FINAL_EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=2,
    shuffle=True,
)


final_prediction_dict = final_model.predict(
    X_all,
    batch_size=BATCH_SIZE,
    verbose=0,
)

final_class_probabilities = final_prediction_dict[
    "class_output"
]
final_class_predictions = np.argmax(
    final_class_probabilities,
    axis=1,
)

print("\nFINAL TRAINING-DATA CLASSIFICATION REPORT")
print(
    classification_report(
        y_class_all,
        final_class_predictions,
        labels=np.arange(N_CLASSES),
        target_names=FINAL_LABELS,
        zero_division=0,
    )
)
print(
    "Final training-data accuracy:",
    accuracy_score(
        y_class_all,
        final_class_predictions,
    ),
)
print(
    "NOTE: this is training-set fit, not an unbiased generalization result. "
    "Use the cross-validation metrics for model performance."
)

pd.DataFrame(final_history.history).to_csv(
    os.path.join(
        FINAL_DIR,
        "final_training_history.csv",
    ),
    index=False,
)

In [ ]:
# ==========================================================
# CELL 15 — save model, labels, metadata, thresholds
# (CHANGED: thresholds dict includes global_normal_baseline,
# local_normal_baselines, and severity multipliers.)
# ==========================================================
KERAS_PATH = os.path.abspath(os.path.join(FINAL_DIR, "pure_tcn_ae_anomaly_classifier.keras"))
H5_PATH = os.path.abspath(os.path.join(FINAL_DIR, "pure_tcn_ae_anomaly_classifier.h5"))
LABELS_PATH = os.path.abspath(os.path.join(FINAL_DIR, "labels.txt"))
METADATA_PATH = os.path.abspath(os.path.join(FINAL_DIR, "metadata.json"))
THRESHOLDS_PATH = os.path.abspath(os.path.join(FINAL_DIR, "thresholds.json"))

final_model.save(KERAS_PATH)
final_model.save(H5_PATH)

with open(LABELS_PATH, "w", encoding="utf-8") as file:
    file.write("\n".join(FINAL_LABELS))

thresholds = {
    "global_reconstruction_fallback": RECON_THRESHOLD,
    "global_normal_baseline": GLOBAL_NORMAL_BASELINE,
    "state_reconstruction_thresholds": STATE_RECON_THRESHOLDS,
    "pwm_bin_edges": PWM_BIN_EDGES.tolist(),
    "pwm_bin_names": PWM_BIN_NAMES,
    "load_proxy_feature": LOAD_PROXY_FEATURE,
    "global_normal_load_split": GLOBAL_NORMAL_LOAD_SPLIT,
    "pwm_load_splits": PWM_LOAD_SPLITS,
    "normal_pwm_reconstruction_thresholds": NORMAL_PWM_RECON_THRESHOLDS,
    "normal_pwm_local_baselines": LOCAL_NORMAL_BASELINES,
    "normal_pwm_load_reconstruction_thresholds": NORMAL_PWM_LOAD_RECON_THRESHOLDS,
    "anomaly_probability_threshold": ANOMALY_PROB_THRESHOLD,
    "class_confidence_threshold": CLASS_CONF_THRESHOLD,
    "severity_mild_multiplier": SEVERITY_MILD_MULTIPLIER,
    "severity_moderate_multiplier": SEVERITY_MODERATE_MULTIPLIER,
    "severity_severe_multiplier": SEVERITY_SEVERE_MULTIPLIER,
}

metadata = {
    "model_name": "pure_tcn_ae_anomaly_classifier",
    "input_shape": [WINDOW_LEN, N_FEATURES],
    "feature_columns": FEATURE_COLUMNS,
    "labels": FINAL_LABELS,
    "safe_labels": SAFE_LABELS,
    "fault_labels": FAULT_LABELS,
    "scale_divisor": SCALE_DIVISOR,
    "window_step": WINDOW_STEP,
    "row_period_sec": ROW_PERIOD_SEC,
    "parameter_count": int(final_model.count_params()),
    "normalization_mean": final_mean.tolist(),
    "normalization_variance": final_variance.tolist(),
    "final_training_epochs": FINAL_EPOCHS,
    "thresholds": thresholds,
}

with open(METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

with open(THRESHOLDS_PATH, "w", encoding="utf-8") as file:
    json.dump(thresholds, file, indent=2)

print("Saved:", KERAS_PATH)
print("Saved:", H5_PATH)
print("Saved:", LABELS_PATH)
print("Saved:", METADATA_PATH)
print("Saved:", THRESHOLDS_PATH)

In [ ]:
# ==========================================================
# CELL 16 — export one-input ONNX for STM32Cube.AI
# ==========================================================
import tf2onnx
import onnx
import onnxruntime as ort

ONNX_PATH = os.path.abspath(
    os.path.join(
        FINAL_DIR,
        "pure_tcn_ae_anomaly_classifier.onnx",
    )
)

input_signature = [
    tf.TensorSpec(
        shape=(
            1,
            WINDOW_LEN,
            N_FEATURES,
        ),
        dtype=tf.float32,
        name="motor_window",
    ),
]

# Remove the Keras Normalization layer from the deployment core and
# recreate normalization with embedded tf.constant values. This avoids
# exporting mean and variance as extra ONNX runtime inputs.
normalization_output = final_model.get_layer(
    "input_normalization"
).output

deployment_core = tf.keras.Model(
    inputs=normalization_output,
    outputs={
        "reconstruction_output":
            final_model.get_layer("reconstruction_output").output,
        "anomaly_output":
            final_model.get_layer("anomaly_output").output,
        "class_output":
            final_model.get_layer("class_output").output,
    },
    name="pure_tcn_deployment_core",
)

export_mean_values = np.asarray(
    final_mean,
    dtype=np.float32,
).reshape(N_FEATURES).tolist()

export_variance_values = np.asarray(
    final_variance,
    dtype=np.float32,
).reshape(N_FEATURES).tolist()


@tf.function(input_signature=input_signature)
def pure_tcn_export_function(motor_window):
    mean_constant = tf.reshape(
        tf.constant(
            export_mean_values,
            dtype=tf.float32,
        ),
        (1, 1, N_FEATURES),
    )
    variance_constant = tf.reshape(
        tf.constant(
            export_variance_values,
            dtype=tf.float32,
        ),
        (1, 1, N_FEATURES),
    )

    normalized_window = (
        motor_window - mean_constant
    ) / tf.sqrt(
        variance_constant + 1.0e-7
    )

    outputs = deployment_core(
        normalized_window,
        training=False,
    )

    return (
        tf.identity(
            outputs["reconstruction_output"],
            name="reconstruction_output",
        ),
        tf.identity(
            outputs["anomaly_output"],
            name="anomaly_output",
        ),
        tf.identity(
            outputs["class_output"],
            name="class_output",
        ),
    )

# from_function is required so embedded normalization is included.
onnx_model, _ = tf2onnx.convert.from_function(
    pure_tcn_export_function,
    input_signature=input_signature,
    opset=13,
    output_path=ONNX_PATH,
)

onnx.checker.check_model(
    onnx.load(ONNX_PATH)
)

onnx_session = ort.InferenceSession(
    ONNX_PATH,
    providers=["CPUExecutionProvider"],
)

onnx_inputs = onnx_session.get_inputs()

if len(onnx_inputs) != 1:
    raise RuntimeError(
        "ONNX export must have exactly one input. Found: "
        f"{[item.name for item in onnx_inputs]}"
    )

verification_input = np.asarray(
    X_all[:1],
    dtype=np.float32,
)

keras_verification = final_model.predict(
    verification_input,
    verbose=0,
)

onnx_values = onnx_session.run(
    None,
    {
        onnx_inputs[0].name:
            verification_input,
    },
)

onnx_by_shape = {
    tuple(value.shape): value
    for value in onnx_values
}

onnx_reconstruction = onnx_by_shape[
    (1, WINDOW_LEN, N_FEATURES)
]
onnx_anomaly = onnx_by_shape[(1, 1)]
onnx_class = onnx_by_shape[(1, N_CLASSES)]

maximum_export_difference = max(
    float(np.max(np.abs(
        keras_verification["reconstruction_output"]
        - onnx_reconstruction
    ))),
    float(np.max(np.abs(
        keras_verification["anomaly_output"]
        - onnx_anomaly
    ))),
    float(np.max(np.abs(
        keras_verification["class_output"]
        - onnx_class
    ))),
)

if maximum_export_difference > 2.0e-4:
    raise RuntimeError(
        "ONNX output differs from Keras by "
        f"{maximum_export_difference}"
    )

print("Saved ONNX:", ONNX_PATH)
print("ONNX input:", onnx_inputs[0].name, onnx_inputs[0].shape)
print("Maximum Keras/ONNX difference:", maximum_export_difference)

print(
    "ONNX size KB:",
    round(
        os.path.getsize(ONNX_PATH) / 1024,
        2,
    ),
)


In [ ]:
# ==========================================================
# CELL 17 — full INT8 TFLite export
# ==========================================================
TFLITE_FLOAT32_PATH = os.path.abspath(
    os.path.join(
        FINAL_DIR,
        "pure_tcn_ae_anomaly_classifier_float32.tflite",
    )
)

TFLITE_INT8_PATH = os.path.abspath(
    os.path.join(
        FINAL_DIR,
        "pure_tcn_ae_anomaly_classifier_int8.tflite",
    )
)
# ----------------------------------------------------------
# Export Float32 TFLite model
# ----------------------------------------------------------
float32_converter = tf.lite.TFLiteConverter.from_keras_model(
    final_model
)

tflite_float32_model = float32_converter.convert()

with open(
    TFLITE_FLOAT32_PATH,
    "wb",
) as file:
    file.write(
        tflite_float32_model
    )

tflite_float32_size_kb = (
    os.path.getsize(
        TFLITE_FLOAT32_PATH
    )
    / 1024
)

print("Saved Float32 TFLite:", TFLITE_FLOAT32_PATH)
print(
    "Float32 TFLite size KB:",
    round(
        tflite_float32_size_kb,
        2,
    ),
)


# ----------------------------------------------------------
# Representative dataset for full INT8 quantization
# ----------------------------------------------------------
def representative_dataset():
    sample_count = min(
        500,
        len(X_all),
    )

    sample_indices = np.linspace(
        0,
        len(X_all) - 1,
        sample_count,
        dtype=np.int64,
    )

    for index in sample_indices:
        yield [
            X_all[
                index:index + 1
            ].astype(np.float32)
        ]


# ----------------------------------------------------------
# Export full INT8 TFLite model
# ----------------------------------------------------------
int8_converter = tf.lite.TFLiteConverter.from_keras_model(
    final_model
)

int8_converter.optimizations = [
    tf.lite.Optimize.DEFAULT
]

int8_converter.representative_dataset = (
    representative_dataset
)

int8_converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

int8_converter.inference_input_type = tf.int8
int8_converter.inference_output_type = tf.int8

tflite_int8_model = int8_converter.convert()

with open(
    TFLITE_INT8_PATH,
    "wb",
) as file:
    file.write(
        tflite_int8_model
    )

tflite_int8_size_kb = (
    os.path.getsize(
        TFLITE_INT8_PATH
    )
    / 1024
)

print("Saved INT8 TFLite:", TFLITE_INT8_PATH)
print(
    "INT8 TFLite size KB:",
    round(
        tflite_int8_size_kb,
        2,
    ),
)


# ----------------------------------------------------------
# Size comparison
# ----------------------------------------------------------
size_reduction_percent = (
    1.0
    - (
        tflite_int8_size_kb
        / tflite_float32_size_kb
    )
) * 100.0

print(
    "INT8 size reduction compared with Float32:",
    round(
        size_reduction_percent,
        2,
    ),
    "%",
)

if tflite_int8_size_kb <= 30.0:
    print(
        "PASS: INT8 model is within the 30 KB target."
    )
else:
    print(
        "WARNING: INT8 model exceeds the 30 KB target."
    )


In [ ]:
# ==========================================================
# CELL-18.1 Export complete TFLite model as one C header file
# ==========================================================
from pathlib import Path
from IPython.display import FileLink, display

TFLITE_MODEL_PATH = Path(
    TFLITE_FLOAT32_PATH
)

HEADER_OUTPUT_PATH = TFLITE_MODEL_PATH.with_name(
    "pure_tcn_3head_model.h"
)

ARRAY_NAME = "g_pure_tcn_3head_model"


def tflite_to_c_header(
    input_path: Path,
    output_path: Path,
    array_name: str,
) -> None:
    if not input_path.exists():
        raise FileNotFoundError(
            f"TFLite model not found:\n{input_path}"
        )

    model_bytes = input_path.read_bytes()

    with output_path.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as header:
        header.write("#ifndef PURE_TCN_3HEAD_MODEL_H\n")
        header.write("#define PURE_TCN_3HEAD_MODEL_H\n\n")

        header.write("#include <stdint.h>\n")
        header.write("#include <stddef.h>\n\n")

        # Alignment is required by TensorFlow Lite Micro.
        header.write("#if defined(__GNUC__)\n")
        header.write("#define TCN_MODEL_ALIGN __attribute__((aligned(16)))\n")
        header.write("#elif defined(__ICCARM__)\n")
        header.write("#define TCN_MODEL_ALIGN _Pragma(\"data_alignment=16\")\n")
        header.write("#else\n")
        header.write("#define TCN_MODEL_ALIGN\n")
        header.write("#endif\n\n")

        header.write(
            f"static const TCN_MODEL_ALIGN uint8_t "
            f"{array_name}[] = {{\n"
        )

        bytes_per_line = 12

        for start in range(
            0,
            len(model_bytes),
            bytes_per_line,
        ):
            chunk = model_bytes[
                start:start + bytes_per_line
            ]

            values = ", ".join(
                f"0x{value:02X}"
                for value in chunk
            )

            header.write(f"    {values}")

            if start + bytes_per_line < len(model_bytes):
                header.write(",")

            header.write("\n")

        header.write("};\n\n")

        header.write(
            f"static const size_t "
            f"{array_name}_len = sizeof({array_name});\n\n"
        )

        header.write("#endif /* PURE_TCN_3HEAD_MODEL_H */\n")

    print("Header created successfully:")
    print(output_path)
    print("Model size:", len(model_bytes), "bytes")


tflite_to_c_header(
    input_path=TFLITE_MODEL_PATH,
    output_path=HEADER_OUTPUT_PATH,
    array_name=ARRAY_NAME,
)

display(
    FileLink(
        str(HEADER_OUTPUT_PATH),
        result_html_prefix="Download header file: ",
    )
)

In [ ]:
# ==========================================================
# CELL 18.2 — standalone float32 C header for the dual-encoder TCN
# No physical motor-off rule; no TensorFlow Lite runtime.
# Requires final_model and Cell 13 calibration variables.
# ==========================================================
from pathlib import Path
import numpy as np
import tensorflow as tf

HEADER_PATH = Path(FINAL_DIR) / 'complete_pure_tcn_3head_model.h'

RECON_BLOCKS = [
    'recon_encoder_tcn_k3', 'recon_encoder_tcn_k5', 'recon_latent_tcn_k9',
    'decoder_tcn_k9', 'decoder_tcn_k5', 'decoder_tcn_k3',
]
SHARED_BLOCKS = ['encoder_tcn_k3', 'encoder_tcn_k5', 'latent_tcn_k9']
ANOMALY_BLOCKS = ['anomaly_tcn_k5', 'anomaly_tcn_k9']
CLASS_BLOCKS = ['class_tcn_k5', 'class_tcn_k9']
ALL_BLOCKS = RECON_BLOCKS + SHARED_BLOCKS + ANOMALY_BLOCKS + CLASS_BLOCKS


def cnum(value):
    v = float(value)
    if not np.isfinite(v):
        raise ValueError(f'Nonfinite model value: {v}')
    s = f'{v:.9g}'
    if '.' not in s and 'e' not in s:
        s += '.0'
    return s + 'f'


def emit_array(out, name, values):
    data = np.asarray(values, dtype=np.float32).reshape(-1)
    out.write(f'static const float {name}[{len(data)}] = {{\n')
    for i in range(0, len(data), 8):
        out.write('  ' + ', '.join(cnum(x) for x in data[i:i + 8]))
        out.write(',\n' if i + 8 < len(data) else '\n')
    out.write('};\n')


def conv_info(name):
    lyr = final_model.get_layer(name)
    if not isinstance(lyr, tf.keras.layers.Conv1D):
        raise TypeError(f'{name} is not a Conv1D layer')
    weights = lyr.get_weights()
    if len(weights) != 2:
        raise ValueError(f'{name}: missing kernel/bias')
    k, b = weights
    return k, b, int(lyr.kernel_size[0]), int(lyr.dilation_rate[0]), int(k.shape[1]), int(k.shape[2])


def emit_conv(out, name):
    k, b, kernel_size, dilation, in_ch, out_ch = conv_info(name)
    assert dilation == 1, (name, dilation)
    emit_array(out, name + '_kernel', k)
    emit_array(out, name + '_bias', b)
    return f'{{{name}_kernel, {name}_bias, {kernel_size}, {in_ch}, {out_ch}}}'


def emit_dense(out, name):
    layer = final_model.get_layer(name)
    k, b = layer.get_weights()
    emit_array(out, name + '_kernel', k)
    emit_array(out, name + '_bias', b)
    return int(k.shape[0]), int(k.shape[1])


# The model from Cell 8 must be the dual-encoder architecture.
for block in ALL_BLOCKS:
    final_model.get_layer(block + '_conv1')
    final_model.get_layer(block + '_conv2')

mean = np.asarray(final_mean, dtype=np.float32).reshape(-1)
variance = np.asarray(final_variance, dtype=np.float32).reshape(-1)
assert len(mean) == len(variance) == N_FEATURES
assert WINDOW_LEN == 30 and N_FEATURES == 14 and N_CLASSES == 7

with HEADER_PATH.open('w', encoding='utf-8', newline='\n') as out:
    out.write('#ifndef COMPLETE_PURE_TCN_3HEAD_MODEL_H\n#define COMPLETE_PURE_TCN_3HEAD_MODEL_H\n')
    out.write('#include <math.h>\n#include <stddef.h>\n#include <stdint.h>\n')
    out.write(f'#define TCN_T {WINDOW_LEN}\n#define TCN_F {N_FEATURES}\n#define TCN_CLASSES {N_CLASSES}\n')
    out.write(f'#define TCN_CHANNELS {max(N_FEATURES, ENCODER_FILTERS_1, ENCODER_FILTERS_2, LATENT_FILTERS, CLASS_DENSE_UNITS, ANOMALY_DENSE_UNITS)}\n')
    out.write(f'#define TCN_INPUT_DIVISOR {cnum(FIRMWARE_INPUT_DIVISOR)}\n')
    out.write(f'#define TCN_CONF_THRESHOLD {cnum(CLASS_CONF_THRESHOLD)}\n')
    out.write(f'#define TCN_ANOM_THRESHOLD {cnum(ANOMALY_PROB_THRESHOLD)}\n')
    out.write(f'#define TCN_FALLBACK_RECON_THRESHOLD {cnum(RECON_THRESHOLD)}\n')
    out.write('typedef struct {float prob[TCN_CLASSES]; float confidence; int class_id; '
              'float anomaly_probability; float mse, recon_threshold; '
              'int binary_anomaly, reconstruction_anomaly, known_fault, final_anomaly;} TCNPrediction;\n')
    emit_array(out, 'tcn_mean', mean)
    emit_array(out, 'tcn_var', variance)
    emit_array(out, 'tcn_pwm_edges', PWM_BIN_EDGES)
    emit_array(out, 'tcn_load_split', [PWM_LOAD_SPLITS[x] for x in PWM_BIN_NAMES])
    emit_array(out, 'tcn_low_thr', [NORMAL_PWM_LOAD_RECON_THRESHOLDS[x]['low_load'] for x in PWM_BIN_NAMES])
    emit_array(out, 'tcn_high_thr', [NORMAL_PWM_LOAD_RECON_THRESHOLDS[x]['high_load'] for x in PWM_BIN_NAMES])
    emit_array(out, 'tcn_state_thr', [STATE_RECON_THRESHOLDS[x] for x in ('motor_off', 'turning_off', 'turning_on')])

    # All 13 residual blocks, including the independent reconstruction encoder.
    out.write('typedef struct {const float *k, *b; int kw, ic, oc;} TCNConv;\n')
    out.write('typedef struct {TCNConv a, b, projection; int has_projection;} TCNBlock;\n')
    for block in ALL_BLOCKS:
        a = emit_conv(out, block + '_conv1')
        b = emit_conv(out, block + '_conv2')
        try:
            final_model.get_layer(block + '_projection')
            p = emit_conv(out, block + '_projection')
            projection_flag = 1
        except ValueError:
            p = '{NULL, NULL, 0, 0, 0}'
            projection_flag = 0
        out.write(f'static const TCNBlock b_{block} = {{{a}, {b}, {p}, {projection_flag}}};\n')

    recon_kernel, recon_bias, recon_kw, _, recon_ic, recon_oc = conv_info('reconstruction_bounded')
    assert recon_kw == 1 and recon_oc == N_FEATURES
    emit_array(out, 'tcn_recon_kernel', recon_kernel)
    emit_array(out, 'tcn_recon_bias', recon_bias)
    an_in, an_out = emit_dense(out, 'anomaly_output')
    cl_in, cl_out = emit_dense(out, 'class_output')
    assert an_out == 1 and cl_out == N_CLASSES
    assert an_in == ANOMALY_DENSE_UNITS and cl_in == CLASS_DENSE_UNITS

    out.write(r'''
static inline float tcn_sigmoid(float x) {
  if (x >= 0) return 1.0f / (1.0f + expf(-x));
  float z = expf(x); return z / (1.0f + z);
}
static inline float tcn_relu(float x) { return x > 0.0f ? x : 0.0f; }
/* Input and output are [30][TCN_CHANNELS]. Unused channels must be zero. */
static inline void tcn_conv(const float input[TCN_T][TCN_CHANNELS],
       float output[TCN_T][TCN_CHANNELS], const TCNConv *c) {
  for (int t=0;t<TCN_T;t++) for (int o=0;o<TCN_CHANNELS;o++) {
    if (o>=c->oc) {output[t][o]=0.0f;continue;}
    float v=c->b[o];
    for(int k=0;k<c->kw;k++) {
      int ti=t-(c->kw-1-k); /* causal left padding */
      if(ti<0) continue;
      for(int i=0;i<c->ic;i++)
        v+=input[ti][i]*c->k[(k*c->ic+i)*c->oc+o];
    }
    output[t][o]=v;
  }
}
static inline void tcn_block(const float input[TCN_T][TCN_CHANNELS],
       float output[TCN_T][TCN_CHANNELS],const TCNBlock *block) {
  float v1[TCN_T][TCN_CHANNELS],v2[TCN_T][TCN_CHANNELS];
  tcn_conv(input,v1,&block->a);
  for(int t=0;t<TCN_T;t++) for(int c=0;c<block->a.oc;c++)
    v1[t][c]=tcn_relu(v1[t][c]);
  tcn_conv(v1,v2,&block->b);
  if(block->has_projection) {
    float projection[TCN_T][TCN_CHANNELS];
    tcn_conv(input,projection,&block->projection);
    for(int t=0;t<TCN_T;t++) for(int c=0;c<TCN_CHANNELS;c++)
      output[t][c] = c<block->b.oc ? tcn_relu(v2[t][c]+projection[t][c]):0.0f;
  } else {
    for(int t=0;t<TCN_T;t++) for(int c=0;c<TCN_CHANNELS;c++)
      output[t][c] = c<block->b.oc ? tcn_relu(v2[t][c]+input[t][c]):0.0f;
  }
}
static inline void tcn_copy(float d[TCN_T][TCN_CHANNELS],
    const float s[TCN_T][TCN_CHANNELS]) {
  for(int t=0;t<TCN_T;t++) for(int c=0;c<TCN_CHANNELS;c++) d[t][c]=s[t][c];
}
static inline void tcn_mean_pool(const float x[TCN_T][TCN_CHANNELS],int channels,float *v){
  for(int c=0;c<channels;c++) {float s=0.0f;
    for(int t=0;t<TCN_T;t++) s+=x[t][c];
    v[c]=s/(float)TCN_T;
  }
}
static inline float tcn_threshold(int class_id,float conf,float pwm,float irms) {
  /* No PWM/current/voltage rule that forces motor_off. */
  if(conf >= TCN_CONF_THRESHOLD && (class_id==1 || class_id==2))
    return tcn_state_thr[class_id];
  int bin=0;
  while(bin<6 && pwm>=tcn_pwm_edges[bin+1]) bin++;
  return irms>=tcn_load_split[bin] ? tcn_high_thr[bin] : tcn_low_thr[bin];
}
/* Outputs three model heads. No physical motor-off override. */
static inline int tcn_predict(const float raw[TCN_T][TCN_F],
    TCNPrediction *result,float reconstruction[TCN_T][TCN_F]) {
  if(!raw || !result || !reconstruction) return -1;
  float x[TCN_T][TCN_CHANNELS]={0}, a[TCN_T][TCN_CHANNELS], b[TCN_T][TCN_CHANNELS];
  float mean_pwm=0.0f,mean_irms=0.0f;
  for(int t=0;t<TCN_T;t++) {
    for(int f=0;f<TCN_F;f++) {
      float physical=raw[t][f]/TCN_INPUT_DIVISOR;
      x[t][f]=(physical-tcn_mean[f])/sqrtf(tcn_var[f]+1e-7f);
    }
    mean_pwm+=raw[t][1]/TCN_INPUT_DIVISOR;
    mean_irms+=raw[t][2]/TCN_INPUT_DIVISOR;
  }
  mean_pwm/=(float)TCN_T;mean_irms/=(float)TCN_T;
  /* Reconstruction path: independent encoder and decoder. */
''')
    for block in RECON_BLOCKS:
        out.write(f'  tcn_block(x,a,&b_{block}); tcn_copy(x,a);\n')
    out.write(f'''  float mse=0.0f;
  for(int t=0;t<TCN_T;t++) for(int f=0;f<TCN_F;f++) {{
    float v=tcn_recon_bias[f];
    for(int c=0;c<{recon_ic};c++) v+=x[t][c]*tcn_recon_kernel[c*TCN_F+f];
    reconstruction[t][f]={cnum(6.0)}*tanhf(v);
    /* reconstruction MSE is measured in normalized feature space */
    float original=(raw[t][f]/TCN_INPUT_DIVISOR-tcn_mean[f])/sqrtf(tcn_var[f]+1e-7f);
    float d=original-reconstruction[t][f];mse+=d*d;
  }}
  result->mse=mse/(float)(TCN_T*TCN_F);
  /* Discriminative path restarts with normalized inputs, not recon latent. */
  for(int t=0;t<TCN_T;t++) for(int c=0;c<TCN_CHANNELS;c++)
    x[t][c] = c<TCN_F ? (raw[t][c]/TCN_INPUT_DIVISOR-tcn_mean[c])/sqrtf(tcn_var[c]+1e-7f):0.0f;
''')
    for block in SHARED_BLOCKS:
        out.write(f'  tcn_block(x,a,&b_{block}); tcn_copy(x,a);\n')
    out.write('  tcn_copy(b,x); /* latent for both discriminative heads */\n')
    for block in ANOMALY_BLOCKS:
        out.write(f'  tcn_block(x,a,&b_{block}); tcn_copy(x,a);\n')
    out.write(f'''  float av[{an_in}]; tcn_mean_pool(x,{an_in},av);
  float logit=anomaly_output_bias[0];
  for(int c=0;c<{an_in};c++) logit+=av[c]*anomaly_output_kernel[c];
  result->anomaly_probability=tcn_sigmoid(logit);
  tcn_copy(x,b);
''')
    for block in CLASS_BLOCKS:
        out.write(f'  tcn_block(x,a,&b_{block}); tcn_copy(x,a);\n')
    out.write(f'''  float features[{cl_in}]; tcn_mean_pool(x,{cl_in},features);
  float maximum=-INFINITY;
  for(int k=0;k<TCN_CLASSES;k++) {{
    float v=class_output_bias[k];
    for(int c=0;c<{cl_in};c++) v+=features[c]*class_output_kernel[c*TCN_CLASSES+k];
    result->prob[k]=v;
    if(v>maximum) maximum=v;
  }}
  float denom=0.0f;
  for(int k=0;k<TCN_CLASSES;k++) {{result->prob[k]=expf(result->prob[k]-maximum);denom+=result->prob[k];}}
  if(!(denom>0.0f) || !isfinite(denom)) return -2;
  result->class_id=0;result->confidence=0.0f;
  for(int k=0;k<TCN_CLASSES;k++) {{
    result->prob[k]/=denom;
    if(result->prob[k]>result->confidence) {{result->confidence=result->prob[k];result->class_id=k;}}
  }}
  result->recon_threshold=tcn_threshold(result->class_id,result->confidence,mean_pwm,mean_irms);
  result->binary_anomaly=result->anomaly_probability>=TCN_ANOM_THRESHOLD;
  result->reconstruction_anomaly=result->mse>result->recon_threshold;
  result->known_fault=(result->class_id>=4 && result->confidence>=TCN_CONF_THRESHOLD);
  result->final_anomaly=(result->binary_anomaly || result->reconstruction_anomaly || result->known_fault);
  return result->class_id;
}}
#endif /* COMPLETE_PURE_TCN_3HEAD_MODEL_H */
''')
print('Created:', HEADER_PATH.resolve())
print('Note: model prediction is never forced to motor_off by PWM/current/voltage.')


In [ ]:
# ==========================================================
# CELL 19 — export firmware parameters header
# (CHANGED: added severity multiplier defines, global baseline
# define, and a local-baselines array.)
# ==========================================================
PARAMS_HEADER_PATH = os.path.abspath(os.path.join(FINAL_DIR, "motor_model_params.h"))


def c_float(value):
    return f"{float(value):.9g}f"


normal_low_load_thresholds = np.asarray([
    NORMAL_PWM_LOAD_RECON_THRESHOLDS[bin_name]["low_load"]
    for bin_name in PWM_BIN_NAMES
], dtype=np.float32)

normal_high_load_thresholds = np.asarray([
    NORMAL_PWM_LOAD_RECON_THRESHOLDS[bin_name]["high_load"]
    for bin_name in PWM_BIN_NAMES
], dtype=np.float32)

pwm_load_split_values = np.asarray([
    PWM_LOAD_SPLITS[bin_name] for bin_name in PWM_BIN_NAMES
], dtype=np.float32)

# ADDED
local_normal_baseline_values = np.asarray([
    LOCAL_NORMAL_BASELINES[bin_name] for bin_name in PWM_BIN_NAMES
], dtype=np.float32)


with open(PARAMS_HEADER_PATH, "w", encoding="utf-8") as file:
    file.write("#ifndef MOTOR_MODEL_PARAMS_H\n")
    file.write("#define MOTOR_MODEL_PARAMS_H\n\n")
    file.write(f"#define MOTOR_MODEL_WINDOW_LEN {WINDOW_LEN}\n")
    file.write(f"#define MOTOR_MODEL_N_FEATURES {N_FEATURES}\n")
    file.write(f"#define MOTOR_MODEL_INPUT_SIZE {WINDOW_LEN * N_FEATURES}\n")
    file.write(f"#define MOTOR_MODEL_N_CLASSES {N_CLASSES}\n")
    file.write(f"#define MOTOR_MODEL_N_PWM_BINS {len(PWM_BIN_NAMES)}\n")
    file.write(
        "#define MOTOR_MODEL_LOAD_PROXY_FEATURE_INDEX "
        f"{FEATURE_COLUMNS.index(LOAD_PROXY_FEATURE)}\n"
    )
    file.write(f"#define MOTOR_MODEL_PWM_FEATURE_INDEX {FEATURE_COLUMNS.index('PWM')}\n")
    file.write(f"#define MOTOR_MODEL_VRMS_FEATURE_INDEX {FEATURE_COLUMNS.index('Vrms')}\n")
    file.write(f"#define MOTOR_MODEL_ANOMALY_THRESHOLD {c_float(ANOMALY_PROB_THRESHOLD)}\n")
    file.write(f"#define MOTOR_MODEL_CLASS_CONF_THRESHOLD {c_float(CLASS_CONF_THRESHOLD)}\n")
    file.write(f"#define MOTOR_MODEL_MOTOR_OFF_PWM_LIMIT {c_float(MOTOR_OFF_PWM_LIMIT)}\n")
    file.write(f"#define MOTOR_MODEL_MOTOR_OFF_IRMS_LIMIT {c_float(MOTOR_OFF_IRMS_LIMIT)}\n")
    file.write(f"#define MOTOR_MODEL_MOTOR_OFF_VRMS_LIMIT {c_float(MOTOR_OFF_VRMS_LIMIT)}\n")
    file.write(f"#define MOTOR_MODEL_RECON_THRESHOLD {c_float(RECON_THRESHOLD)}\n")
    # ADDED
    file.write(f"#define MOTOR_MODEL_GLOBAL_NORMAL_BASELINE {c_float(GLOBAL_NORMAL_BASELINE)}\n")
    file.write(f"#define MOTOR_MODEL_SEVERITY_MILD_MULTIPLIER {c_float(SEVERITY_MILD_MULTIPLIER)}\n")
    file.write(f"#define MOTOR_MODEL_SEVERITY_MODERATE_MULTIPLIER {c_float(SEVERITY_MODERATE_MULTIPLIER)}\n")
    file.write(f"#define MOTOR_MODEL_SEVERITY_SEVERE_MULTIPLIER {c_float(SEVERITY_SEVERE_MULTIPLIER)}\n\n")

    file.write(f"static const float MOTOR_MODEL_NORMALIZATION_MEAN[{N_FEATURES}] = {{\n    ")
    file.write(", ".join(c_float(v) for v in final_mean))
    file.write("\n};\n\n")

    file.write(f"static const float MOTOR_MODEL_NORMALIZATION_VARIANCE[{N_FEATURES}] = {{\n    ")
    file.write(", ".join(c_float(v) for v in final_variance))
    file.write("\n};\n\n")

    file.write(f"static const float MOTOR_MODEL_PWM_BIN_EDGES[{len(PWM_BIN_EDGES)}] = {{\n    ")
    file.write(", ".join(c_float(v) for v in PWM_BIN_EDGES))
    file.write("\n};\n\n")

    file.write(f"static const float MOTOR_MODEL_PWM_LOAD_SPLITS[{len(PWM_BIN_NAMES)}] = {{\n    ")
    file.write(", ".join(c_float(v) for v in pwm_load_split_values))
    file.write("\n};\n\n")

    file.write(f"static const float MOTOR_MODEL_NORMAL_LOW_LOAD_THRESHOLDS[{len(PWM_BIN_NAMES)}] = {{\n    ")
    file.write(", ".join(c_float(v) for v in normal_low_load_thresholds))
    file.write("\n};\n\n")

    file.write(f"static const float MOTOR_MODEL_NORMAL_HIGH_LOAD_THRESHOLDS[{len(PWM_BIN_NAMES)}] = {{\n    ")
    file.write(", ".join(c_float(v) for v in normal_high_load_thresholds))
    file.write("\n};\n\n")

    # ADDED
    file.write(f"static const float MOTOR_MODEL_LOCAL_NORMAL_BASELINES[{len(PWM_BIN_NAMES)}] = {{\n    ")
    file.write(", ".join(c_float(v) for v in local_normal_baseline_values))
    file.write("\n};\n\n")

    file.write(f"#define MOTOR_MODEL_MOTOR_OFF_RECON_THRESHOLD {c_float(STATE_RECON_THRESHOLDS['motor_off'])}\n")
    file.write(f"#define MOTOR_MODEL_TURNING_OFF_RECON_THRESHOLD {c_float(STATE_RECON_THRESHOLDS['turning_off'])}\n")
    file.write(f"#define MOTOR_MODEL_TURNING_ON_RECON_THRESHOLD {c_float(STATE_RECON_THRESHOLDS['turning_on'])}\n\n")

    file.write(f"static const char *const MOTOR_MODEL_LABELS[{N_CLASSES}] = {{\n")
    for label in FINAL_LABELS:
        file.write(f'    "{label}",\n')
    file.write("};\n\n")
    file.write("#endif /* MOTOR_MODEL_PARAMS_H */\n")

print("Saved firmware parameters header:", PARAMS_HEADER_PATH)

In [ ]:
# ==========================================================
# CELL 20 — verify TFLite model interfaces
# ==========================================================
interpreter = tf.lite.Interpreter(
    model_path=TFLITE_INT8_PATH
)

interpreter.allocate_tensors()

print("TFLite input details:")

for detail in interpreter.get_input_details():
    print(detail)

print("TFLite output details:")

for detail in interpreter.get_output_details():
    print(detail)

In [ ]:
# ==========================================================
# CELL 21 — manual classification test
# (CHANGED: includes severity_ratio/severity_label/recommended_action
# in the output + a 5th chart plotting severity over time.)
# ==========================================================
def manual_classification_prediction(test_file):
    model = tf.keras.models.load_model(KERAS_PATH, compile=False)

    test_df = read_one(test_file)

    X_test, starts = make_windows_from_array(test_df.to_numpy(dtype=np.float32))

    if X_test is None or len(X_test) == 0:
        raise RuntimeError(
            "The selected file does not contain enough valid rows for one 30-row window."
        )

    X_test_normalized = normalize_windows_numpy(X_test, final_mean, final_variance)

    prediction_dict = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

    reconstructed = prediction_dict["reconstruction_output"]
    anomaly_probabilities = prediction_dict["anomaly_output"].reshape(-1)
    class_probabilities = prediction_dict["class_output"]

    reconstruction_mse = np.mean(
        np.square(reconstructed - X_test_normalized), axis=(1, 2),
    )

    pwm_feature_index = FEATURE_COLUMNS.index("PWM")
    irms_feature_index = FEATURE_COLUMNS.index(LOAD_PROXY_FEATURE)
    vrms_feature_index = FEATURE_COLUMNS.index("Vrms")

    window_mean_pwm = np.mean(X_test[:, :, pwm_feature_index], axis=1).astype(np.float32)
    window_mean_irms = np.mean(X_test[:, :, irms_feature_index], axis=1).astype(np.float32)
    window_mean_vrms = np.mean(X_test[:, :, vrms_feature_index], axis=1).astype(np.float32)

    preliminary_class_predictions = np.argmax(class_probabilities, axis=1).astype(np.int64)
    preliminary_class_confidence = np.max(class_probabilities, axis=1).astype(np.float32)

    physical_motor_off_flags = np.zeros(len(X_test), dtype=bool)

    window_reconstruction_thresholds = np.asarray([
        get_conditional_reconstruction_threshold(class_id, conf, pwm, irms, vrms)
        for class_id, conf, pwm, irms, vrms in zip(
            preliminary_class_predictions, preliminary_class_confidence,
            window_mean_pwm, window_mean_irms, window_mean_vrms,
        )
    ], dtype=np.float32)

    (
        class_predictions,
        class_confidence,
        binary_anomaly_flags,
        reconstruction_anomaly_flags,
        anomaly_flags,
        final_decisions,
        local_baselines,
        severity_ratios,
        severity_labels,
        recommended_actions,
    ) = fuse_decisions(
        class_probabilities,
        anomaly_probabilities,
        reconstruction_mse,
        window_reconstruction_thresholds,
        physical_motor_off_flags,
        window_mean_pwm,
    )

    result = pd.DataFrame({
        "window": np.arange(len(X_test)),
        "start_row": starts,
        "time_sec": (starts + WINDOW_LEN / 2.0) * ROW_PERIOD_SEC,
        "mean_pwm": window_mean_pwm,
        "mean_irms": window_mean_irms,
        "mean_vrms": window_mean_vrms,
        "physical_motor_off": physical_motor_off_flags.astype(np.int64),
        "pred_class_id": class_predictions,
        "pred_label": [FINAL_LABELS[i] for i in class_predictions],
        "class_confidence": class_confidence,
        "anomaly_probability": anomaly_probabilities,
        "binary_anomaly_flag": binary_anomaly_flags.astype(np.int64),
        "reconstruction_mse": reconstruction_mse,
        "reconstruction_threshold": window_reconstruction_thresholds,
        "local_baseline": local_baselines,
        "severity_ratio": severity_ratios,
        "severity_label": severity_labels,
        "recommended_action": recommended_actions,
        "reconstruction_anomaly_flag": reconstruction_anomaly_flags.astype(np.int64),
        "anomaly_flag": anomaly_flags.astype(np.int64),
        "final_decision": final_decisions,
    })

    for class_id, label in enumerate(FINAL_LABELS):
        result[f"probability_{label}"] = class_probabilities[:, class_id]

    majority_class_id = int(result["pred_class_id"].value_counts().idxmax())
    majority_label = FINAL_LABELS[majority_class_id]
    majority_final_decision = result["final_decision"].value_counts().idxmax()
    majority_severity = result["severity_label"].value_counts().idxmax()

    stem = Path(test_file).stem
    output_csv = os.path.join(MANUAL_TEST_DIR, f"manual_test_{stem}.csv")
    result.to_csv(output_csv, index=False)

    print("\n" + "=" * 70)
    print("MANUAL PURE 3-HEAD TCN TEST")
    print("=" * 70)
    print("File:", test_file)
    print("Total windows:", len(result))
    print("Majority class:", majority_label)
    print("Majority final decision:", majority_final_decision)
    print("Majority severity tier:", majority_severity)
    print("Mean class confidence:", float(np.mean(class_confidence)))
    print("Mean anomaly probability:", float(np.mean(anomaly_probabilities)))
    print("Mean reconstruction MSE:", float(np.mean(reconstruction_mse)))
    print("Anomalous windows:", int(np.sum(anomaly_flags)), "/", len(result))
    print("Saved CSV:", output_csv)

    print("\nFinal decisions:")
    print(result["final_decision"].value_counts())

    print("\nSeverity tiers:")
    print(result["severity_label"].value_counts())

    # CHART 1 — multi-class classification head
    plt.figure(figsize=(14, 7))
    for class_id, label in enumerate(FINAL_LABELS):
        plt.plot(result["time_sec"], class_probabilities[:, class_id], label=label, linewidth=1.5)
    plt.axhline(y=CLASS_CONF_THRESHOLD, linestyle="--", label=f"Confidence threshold ({CLASS_CONF_THRESHOLD:.2f})")
    plt.xlabel("Time [s]"); plt.ylabel("Class probability"); plt.ylim(0.0, 1.05)
    plt.title(f"Classification head probabilities — {stem}")
    plt.legend(loc="upper right", ncol=2); plt.grid(True, alpha=0.3); plt.tight_layout()
    class_head_path = os.path.join(MANUAL_TEST_DIR, f"manual_test_{stem}_class_head.png")
    plt.savefig(class_head_path, dpi=180); plt.show(); plt.close()
    print("Saved classification-head chart:", class_head_path)

    # CHART 2 — binary anomaly head
    plt.figure(figsize=(14, 5))
    plt.plot(result["time_sec"], anomaly_probabilities, label="Anomaly probability", linewidth=2)
    plt.axhline(y=ANOMALY_PROB_THRESHOLD, linestyle="--", label=f"Anomaly threshold ({ANOMALY_PROB_THRESHOLD:.2f})")
    plt.fill_between(result["time_sec"], 0, 1, where=(anomaly_probabilities >= ANOMALY_PROB_THRESHOLD), alpha=0.15, label="Anomaly detected")
    plt.xlabel("Time [s]"); plt.ylabel("Anomaly probability"); plt.ylim(0.0, 1.05)
    plt.title(f"Binary anomaly head — {stem}")
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
    anomaly_head_path = os.path.join(MANUAL_TEST_DIR, f"manual_test_{stem}_anomaly_head.png")
    plt.savefig(anomaly_head_path, dpi=180); plt.show(); plt.close()
    print("Saved anomaly-head chart:", anomaly_head_path)

    # CHART 3 — reconstruction autoencoder head
    plt.figure(figsize=(14, 5))
    plt.plot(result["time_sec"], reconstruction_mse, label="Reconstruction MSE", linewidth=2)
    plt.plot(result["time_sec"], window_reconstruction_thresholds, linestyle="--", linewidth=2, label="State/PWM/load-conditioned threshold")
    plt.fill_between(result["time_sec"], 0, reconstruction_mse, where=(reconstruction_mse > window_reconstruction_thresholds), alpha=0.15, label="Reconstruction anomaly")
    plt.xlabel("Time [s]"); plt.ylabel("Reconstruction MSE")
    plt.title(f"Autoencoder reconstruction head — {stem}")
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
    reconstruction_head_path = os.path.join(MANUAL_TEST_DIR, f"manual_test_{stem}_reconstruction_head.png")
    plt.savefig(reconstruction_head_path, dpi=180); plt.show(); plt.close()
    print("Saved reconstruction-head chart:", reconstruction_head_path)

    # CHART 4 — predicted class timeline
    plt.figure(figsize=(14, 5))
    plt.step(result["time_sec"], result["pred_class_id"], where="mid", linewidth=2)
    plt.yticks(np.arange(N_CLASSES), FINAL_LABELS)
    plt.xlabel("Time [s]"); plt.ylabel("Predicted class")
    plt.title(f"Classification result timeline — {stem}")
    plt.grid(True, alpha=0.3); plt.tight_layout()
    timeline_path = os.path.join(MANUAL_TEST_DIR, f"manual_test_{stem}_class_timeline.png")
    plt.savefig(timeline_path, dpi=180); plt.show(); plt.close()
    print("Saved class timeline:", timeline_path)

    # CHART 5 (ADDED) — severity ratio over time
    plt.figure(figsize=(14, 5))
    plt.plot(result["time_sec"], result["severity_ratio"], linewidth=2, color="darkorange", label="Severity ratio (MSE / local baseline)")
    for level, color in [
        (SEVERITY_MILD_MULTIPLIER, "gold"),
        (SEVERITY_MODERATE_MULTIPLIER, "orange"),
        (SEVERITY_SEVERE_MULTIPLIER, "red"),
    ]:
        plt.axhline(y=level, linestyle="--", color=color, alpha=0.7)
    plt.xlabel("Time [s]"); plt.ylabel("Severity ratio")
    plt.title(f"Reconstruction severity — {stem}")
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
    severity_path = os.path.join(MANUAL_TEST_DIR, f"manual_test_{stem}_severity.png")
    plt.savefig(severity_path, dpi=180); plt.show(); plt.close()
    print("Saved severity chart:", severity_path)

    return result

In [ ]:
# ==========================================================
# CELL 22 — select manual test file
# ==========================================================
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)

TEST_FILE = filedialog.askopenfilename(
    title="Select motor CSV or Excel file",
    filetypes=[
        ("Excel files", "*.xlsx"),
        ("CSV files", "*.csv"),
        ("All files", "*.*"),
    ],
)

root.destroy()

if TEST_FILE:
    manual_result = manual_classification_prediction(
        TEST_FILE
    )
else:
    print("No file selected.")

In [ ]:
# ==========================================================
# CELL 23 — Manual test with the DEPLOYED 09-10 model (standalone)
# Uses ONLY the saved .tflite + the thresholds flashed on the STM32
# (motor_model_params.h of DANN_HYBRID). Needs no other cell.
# ==========================================================

%pip install numpy pandas matplotlib tensorflow openpyxl

import os, numpy as np, pandas as pd, tensorflow as tf
import matplotlib.pyplot as plt
from pathlib import Path

DEPLOYED_TFLITE = r"outputs/pure_tcn_ae_classifier_3head_output/final_model/pure_tcn_ae_anomaly_classifier_float32.tflite"
OUT_DIR = r"deployed_09_10_manual_tests"
os.makedirs(OUT_DIR, exist_ok=True)

# ---- exactly as on the STM32 (DANN_HYBRID/motor_model_params.h) ----
D_COLS = ["T_slope_10s","PWM","Irms","Vrms","Current_Variance","Current_Crest_Factor",
          "Vib_RMS","Vib_Min","Vib_Q1","Vib_Median","Vib_Q3","Vib_Max","Vib_Variance","Vib_Skewness"]
D_LABELS = ["motor_off","turning_off","turning_on","normal",
            "electrical_fault","mechanical_fault","thermal_fault"]
D_WIN, D_STEP, D_ROW_SEC, D_SCALE = 30, 10, 0.015, 1_000_000.0
D_ANOMALY_THR, D_CONF_THR = 0.65, 0.65
D_MEAN = np.array([0.0321703702,0.423528224,0.282306701,9.24939823,0.082554318,2.60076427,0.60639751,
                   0.415869385,0.536268353,0.591035485,0.6577034,0.789046824,0.0149320941,0.116665579],np.float32)
D_VAR  = np.array([0.0165831614,0.102087505,0.12657325,23.7579765,0.0455393344,0.324492335,0.00562078087,
                   0.017659558,0.00280927052,0.00374691444,0.0164413601,0.0499093868,0.000539952423,0.593476653],np.float32)
D_PWM_EDGES = np.array([0.0,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0],np.float32)
D_NORMAL_THR = np.array([0.386128217,0.114676803,0.0942992568,0.167960867,0.322025448,0.750934541,0.442112386,0.653334022],np.float32)
D_STATE_THR = {0: 4.03096792, 1: 3.3466177, 2: 6.31182035}

def deployed_manual_test(test_file):
    df = pd.read_excel(test_file) if str(test_file).lower().endswith(".xlsx") else pd.read_csv(test_file)
    X = (df[D_COLS].apply(pd.to_numeric, errors="coerce") / D_SCALE)
    X["PWM"] = X["PWM"].clip(upper=1.0)
    X = X.replace([np.inf, -np.inf], np.nan).dropna().to_numpy(np.float32)
    starts = np.arange(0, len(X) - D_WIN + 1, D_STEP)
    if len(starts) == 0:
        raise RuntimeError("File shorter than one 30-row window.")
    W = np.stack([X[s:s + D_WIN] for s in starts])

    it = tf.lite.Interpreter(model_path=DEPLOYED_TFLITE); it.allocate_tensors()
    inp = it.get_input_details()[0]
    rec, logit, prob = [], [], []
    for w in W:
        it.set_tensor(inp["index"], w[None]); it.invoke()
        out = {tuple(d["shape"]): it.get_tensor(d["index"])[0] for d in it.get_output_details()}
        rec.append(out[(1, 30, 14)]); logit.append(out[(1, 1)][0]); prob.append(out[(1, 7)])
    rec, logit, prob = np.array(rec), np.array(logit), np.array(prob)

    a_prob = 1.0 / (1.0 + np.exp(-logit))                               # logit -> probability
    norm = (W - D_MEAN) / np.sqrt(D_VAR + 1e-7)
    mse = np.mean((rec - norm) ** 2, axis=(1, 2))
    cls = prob.argmax(1); conf = prob.max(1)
    pwm = W[:, :, 1].mean(1); irms = W[:, :, 2].mean(1)
    pbin = np.clip(np.searchsorted(D_PWM_EDGES, pwm, side="right") - 1, 0, 7)
    thr = np.array([D_STATE_THR.get(int(c), D_NORMAL_THR[b]) for c, b in zip(cls, pbin)])

    bin_flag = a_prob >= D_ANOMALY_THR
    rec_flag = mse > thr
    known = (cls >= 4) & (conf >= D_CONF_THR)
    final = np.where(known, np.array(D_LABELS, object)[cls],
             np.where(bin_flag | rec_flag, "unknown_fault",
             np.where(conf >= D_CONF_THR, np.array(D_LABELS, object)[cls], "uncertain_safe")))

    t = (starts + D_WIN / 2) * D_ROW_SEC
    res = pd.DataFrame({"time_sec": t, "mean_pwm": pwm, "mean_irms": irms,
                        "pred_label": np.array(D_LABELS, object)[cls], "class_confidence": conf,
                        "anomaly_logit": logit, "anomaly_probability": a_prob,
                        "reconstruction_mse": mse, "reconstruction_threshold": thr,
                        "binary_flag": bin_flag.astype(int), "recon_flag": rec_flag.astype(int),
                        "final_decision": final})
    for i, l in enumerate(D_LABELS):
        res[f"probability_{l}"] = prob[:, i]

    stem = Path(test_file).stem
    csv = os.path.join(OUT_DIR, f"manual_test_{stem}.csv")
    res.to_csv(csv, index=False)
    anomaly_flags = (known | bin_flag | rec_flag)

    print("\n" + "=" * 70)
    print("MANUAL PURE MLP ANN 3-HEAD TEST  (DEPLOYED 09-10 MODEL)")
    print("=" * 70)
    print("File:", test_file)
    print("Total windows:", len(res))
    print("Majority class:", res["pred_label"].value_counts().idxmax())
    print("Majority final decision:", res["final_decision"].value_counts().idxmax())
    print("Mean class confidence:", float(np.mean(conf)))
    print("Mean anomaly probability:", float(np.mean(a_prob)))
    print("Mean reconstruction MSE:", float(np.mean(mse)))
    print("Anomalous windows:", int(np.sum(anomaly_flags)), "/", len(res))
    print("Saved CSV:", csv)
    print("\nFinal decisions:")
    print(res["final_decision"].value_counts())

    def _save(name):
        p = os.path.join(OUT_DIR, f"manual_test_{stem}_{name}.png")
        plt.tight_layout(); plt.savefig(p, dpi=180); plt.show(); plt.close()
        return p

    # CHART 1 — classification head
    plt.figure(figsize=(14, 7))
    for i, l in enumerate(D_LABELS):
        plt.plot(t, prob[:, i], label=l, linewidth=1.5)
    plt.axhline(y=D_CONF_THR, linestyle="--", label=f"Confidence threshold ({D_CONF_THR:.2f})")
    plt.xlabel("Time [s]"); plt.ylabel("Class probability"); plt.ylim(0.0, 1.05)
    plt.title(f"Classification head probabilities — {stem}")
    plt.legend(loc="upper right", ncol=2); plt.grid(True, alpha=0.3)
    print("Saved classification chart:", _save("class_head"))

    # CHART 2 — binary anomaly head
    plt.figure(figsize=(14, 5))
    plt.plot(t, a_prob, label="Anomaly probability", linewidth=2)
    plt.axhline(y=D_ANOMALY_THR, linestyle="--", label=f"Anomaly threshold ({D_ANOMALY_THR:.3f})")
    plt.fill_between(t, 0, 1, where=(a_prob >= D_ANOMALY_THR), alpha=0.15, label="Anomaly detected")
    plt.xlabel("Time [s]"); plt.ylabel("Anomaly probability"); plt.ylim(0.0, 1.05)
    plt.title(f"Binary anomaly head — {stem}"); plt.legend(); plt.grid(True, alpha=0.3)
    print("Saved anomaly chart:", _save("anomaly_head"))

    # CHART 3 — autoencoder reconstruction head
    plt.figure(figsize=(14, 5))
    plt.plot(t, mse, label="Reconstruction MSE", linewidth=2)
    plt.plot(t, thr, linestyle="--", linewidth=2, label="State/PWM-conditioned threshold")
    plt.fill_between(t, 0, mse, where=(mse > thr), alpha=0.15, label="Reconstruction anomaly")
    plt.xlabel("Time [s]"); plt.ylabel("Reconstruction MSE")
    plt.title(f"Autoencoder reconstruction head — {stem}"); plt.legend(); plt.grid(True, alpha=0.3)
    print("Saved reconstruction chart:", _save("reconstruction_head"))

    # CHART 4 — predicted class timeline
    plt.figure(figsize=(14, 5))
    plt.step(t, cls, where="mid", linewidth=2)
    plt.yticks(np.arange(len(D_LABELS)), D_LABELS)
    plt.xlabel("Time [s]"); plt.ylabel("Predicted class")
    plt.title(f"Classification result timeline — {stem}"); plt.grid(True, alpha=0.3)
    print("Saved class timeline:", _save("class_timeline"))

    return res   

# ---- pick a file and run ----
TEST_FILE_23 = ""          # paste a path here, or leave empty for the file dialog
if not TEST_FILE_23:
    import tkinter as tk
    from tkinter import filedialog
    _r = tk.Tk(); _r.withdraw(); _r.attributes("-topmost", True)
    TEST_FILE_23 = filedialog.askopenfilename(title="Select motor CSV or Excel file",
        filetypes=[("Excel files", "*.xlsx"), ("CSV files", "*.csv"), ("All files", "*.*")])
    _r.destroy()
deployed_result = deployed_manual_test(TEST_FILE_23) if TEST_FILE_23 else print("No file selected.")

In [ ]:
# ==========================================================
# CELL 24 — STANDALONE PURE TCN 3-HEAD MANUAL TEST
# Run ONLY this cell — no previous cells required.
# No training, no recalibration, no model modifications.
# ==========================================================

import json
from pathlib import Path
import tkinter as tk
from tkinter import filedialog

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf


# ==========================================================
# 1. FIND SAVED TCN MODEL
# ==========================================================

OUTPUT_ROOT = Path.cwd() / "pure_tcn_ae_classifier_3head_output"
MODEL_DIR = OUTPUT_ROOT / "final_model"

MODEL_PATH = MODEL_DIR / "pure_tcn_ae_anomaly_classifier.keras"
METADATA_PATH = MODEL_DIR / "metadata.json"
THRESHOLDS_PATH = MODEL_DIR / "thresholds.json"


def select_file(title, filetypes, initialdir=None):
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    path = filedialog.askopenfilename(
        title=title,
        initialdir=str(initialdir or Path.cwd()),
        filetypes=filetypes
    )

    root.destroy()
    return Path(path) if path else None


if not MODEL_PATH.is_file():
    print("Saved TCN model not found in default directory.")

    MODEL_PATH = select_file(
        "Select trained TCN Keras model",
        [("Keras model", "*.keras")]
    )

    if MODEL_PATH is None:
        raise RuntimeError("No TCN model selected.")

    MODEL_DIR = MODEL_PATH.parent
    OUTPUT_ROOT = MODEL_DIR.parent
    METADATA_PATH = MODEL_DIR / "metadata.json"
    THRESHOLDS_PATH = MODEL_DIR / "thresholds.json"


if not METADATA_PATH.is_file():
    raise FileNotFoundError(METADATA_PATH)

with open(METADATA_PATH, encoding="utf-8") as f:
    meta = json.load(f)

if THRESHOLDS_PATH.is_file():
    with open(THRESHOLDS_PATH, encoding="utf-8") as f:
        thresholds = json.load(f)
else:
    thresholds = meta["thresholds"]


# ==========================================================
# 2. LOAD CONFIGURATION AND TRAINED MODEL
# ==========================================================

FEATURES = meta["feature_columns"]
LABELS = meta["labels"]
SAFE_LABELS = meta["safe_labels"]
FAULT_LABELS = meta["fault_labels"]

WIN, NF = map(int, meta["input_shape"])
STEP = int(meta["window_step"])
ROW_SEC = float(meta["row_period_sec"])
SCALE = float(meta["scale_divisor"])

MEAN = np.asarray(meta["normalization_mean"], np.float32)
VAR = np.asarray(meta["normalization_variance"], np.float32)

ANOM_THR = float(thresholds["anomaly_probability_threshold"])
CLASS_THR = float(thresholds["class_confidence_threshold"])

GLOBAL_RECON_THR = float(
    thresholds["global_reconstruction_fallback"]
)
GLOBAL_BASELINE = float(thresholds["global_normal_baseline"])

STATE_THRESHOLDS = thresholds["state_reconstruction_thresholds"]

PWM_EDGES = np.asarray(
    thresholds["pwm_bin_edges"], dtype=np.float32
)
PWM_NAMES = thresholds["pwm_bin_names"]
PWM_SPLITS = thresholds["pwm_load_splits"]

NORMAL_PWM_THRESHOLDS = thresholds[
    "normal_pwm_reconstruction_thresholds"
]
NORMAL_LOAD_THRESHOLDS = thresholds[
    "normal_pwm_load_reconstruction_thresholds"
]
LOCAL_BASELINES = thresholds["normal_pwm_local_baselines"]

GLOBAL_LOAD_SPLIT = float(
    thresholds["global_normal_load_split"]
)

MILD = float(thresholds["severity_mild_multiplier"])
MODERATE = float(thresholds["severity_moderate_multiplier"])
SEVERE = float(thresholds["severity_severe_multiplier"])

BATCH_SIZE = 128

print("Loading trained TCN model...")

model = tf.keras.models.load_model(
    str(MODEL_PATH), compile=False
)

print("TCN model loaded successfully.")
print("Model:", MODEL_PATH)
print("Input:", WIN, "x", NF)


# ==========================================================
# 3. SELECT MANUAL TEST FILE
# ==========================================================

TEST_FILE = select_file(
    "Select motor CSV or Excel file for TCN manual test",
    [
        ("Motor data files", "*.xlsx *.csv"),
        ("Excel files", "*.xlsx"),
        ("CSV files", "*.csv"),
        ("All files", "*.*")
    ]
)

if TEST_FILE is None:
    raise RuntimeError("Manual test cancelled.")

RESULT_DIR = OUTPUT_ROOT / "manual_test_results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

stem = TEST_FILE.stem

print("\nSelected file:", TEST_FILE)
print("Results directory:", RESULT_DIR)


# ==========================================================
# 4. READ AND PREPROCESS DATA — SAME AS TCN CELL 4
# ==========================================================

if TEST_FILE.suffix.lower() == ".xlsx":
    df = pd.read_excel(TEST_FILE)
elif TEST_FILE.suffix.lower() == ".csv":
    df = pd.read_csv(TEST_FILE)
else:
    raise ValueError("Only CSV and XLSX files supported.")

missing = [c for c in FEATURES if c not in df.columns]

if missing:
    raise ValueError(f"Missing feature columns: {missing}")

data = df[FEATURES].apply(pd.to_numeric, errors="coerce")
data = data / SCALE

# Same PWM clipping as original TCN preprocessing.
data["PWM"] = data["PWM"].clip(upper=1.0)

data = data.replace([np.inf, -np.inf], np.nan)
data = data.dropna().reset_index(drop=True)

X = data.to_numpy(dtype=np.float32)

if len(X) < WIN:
    raise ValueError(
        f"Only {len(X)} valid rows; minimum {WIN} required."
    )


# ==========================================================
# 5. CREATE TCN INPUT WINDOWS
# ==========================================================

starts = np.arange(
    0, len(X) - WIN + 1, STEP, dtype=np.int64
)

windows = np.stack([
    X[s:s + WIN] for s in starts
]).astype(np.float32)

print("Valid rows:", len(X))
print("Total windows:", len(windows))


# ==========================================================
# 6. RUN TCN 3-HEAD INFERENCE
# ==========================================================

predictions = model.predict(
    windows,
    batch_size=BATCH_SIZE,
    verbose=0
)

if isinstance(predictions, dict):
    outputs = predictions
else:
    outputs = {}

    for value in predictions:
        value = np.asarray(value)

        if value.shape[1:] == (WIN, NF):
            outputs["reconstruction_output"] = value
        elif value.shape[1:] == (len(LABELS),):
            outputs["class_output"] = value
        elif value.shape[1:] == (1,):
            outputs["anomaly_output"] = value

required = {
    "reconstruction_output",
    "anomaly_output",
    "class_output"
}

if set(outputs) != required:
    raise RuntimeError(
        f"TCN output mapping failed: {list(outputs)}"
    )

reconstructed = np.asarray(
    outputs["reconstruction_output"], np.float32
)
anomaly_probability = np.asarray(
    outputs["anomaly_output"], np.float32
).reshape(-1)
class_probability = np.asarray(
    outputs["class_output"], np.float32
)

# The saved TCN model already applies sigmoid to anomaly_output.
# Do NOT apply sigmoid again.


# ==========================================================
# 7. CALCULATE RECONSTRUCTION MSE
# ==========================================================

normalized = (
    (windows - MEAN.reshape(1, 1, NF))
    / np.sqrt(VAR.reshape(1, 1, NF) + 1e-7)
)

reconstruction_mse = np.mean(
    np.square(reconstructed - normalized),
    axis=(1, 2)
).astype(np.float32)

mean_pwm = windows[:, :, FEATURES.index("PWM")].mean(axis=1)
mean_irms = windows[:, :, FEATURES.index("Irms")].mean(axis=1)
mean_vrms = windows[:, :, FEATURES.index("Vrms")].mean(axis=1)

class_id = np.argmax(class_probability, axis=1).astype(np.int64)
class_confidence = np.max(class_probability, axis=1)


# ==========================================================
# 8. ORIGINAL TCN CONDITIONAL RECONSTRUCTION THRESHOLD
# ==========================================================

def pwm_bin_index(pwm):
    return int(np.clip(
        np.searchsorted(PWM_EDGES, pwm, side="right") - 1,
        0, len(PWM_NAMES) - 1
    ))


def reconstruction_threshold(cid, confidence, pwm, irms, vrms):
    label = LABELS[int(cid)]

    # Exactly as original TCN Cell 11:
    # state threshold only for confident turning_on/turning_off.
    if (
        label in ["turning_off", "turning_on"]
        and confidence >= CLASS_THR
    ):
        return float(STATE_THRESHOLDS[label])

    bin_name = PWM_NAMES[pwm_bin_index(pwm)]

    load_split = float(
        PWM_SPLITS.get(bin_name, GLOBAL_LOAD_SPLIT)
    )

    load_state = (
        "high_load" if irms >= load_split
        else "low_load"
    )

    return float(
        NORMAL_LOAD_THRESHOLDS.get(bin_name, {}).get(
            load_state,
            NORMAL_PWM_THRESHOLDS.get(
                bin_name, GLOBAL_RECON_THR
            )
        )
    )


recon_thresholds = np.asarray([
    reconstruction_threshold(cid, conf, pwm, irms, vrms)
    for cid, conf, pwm, irms, vrms in zip(
        class_id,
        class_confidence,
        mean_pwm,
        mean_irms,
        mean_vrms
    )
], dtype=np.float32)


# ==========================================================
# 9. ORIGINAL TCN THREE-HEAD DECISION FUSION
# ==========================================================

binary_flags = anomaly_probability >= ANOM_THR
reconstruction_flags = reconstruction_mse > recon_thresholds

known_fault_flags = (
    np.isin(class_id, [
        LABELS.index(label) for label in FAULT_LABELS
    ])
    & (class_confidence >= CLASS_THR)
)

anomaly_flags = (
    binary_flags
    | reconstruction_flags
    | known_fault_flags
)

# Original TCN uses no physical motor-off override.
physical_motor_off_flags = np.zeros(len(windows), dtype=bool)

final_decisions = []

for i in range(len(class_id)):
    label = LABELS[int(class_id[i])]

    if known_fault_flags[i]:
        decision = label
    elif binary_flags[i] or reconstruction_flags[i]:
        decision = "unknown_fault"
    elif label in SAFE_LABELS and class_confidence[i] >= CLASS_THR:
        decision = label
    else:
        decision = "uncertain_safe"

    final_decisions.append(decision)


# ==========================================================
# 10. SEVERITY SCORING
# ==========================================================

local_baselines = np.asarray([
    float(LOCAL_BASELINES.get(
        PWM_NAMES[pwm_bin_index(pwm)],
        GLOBAL_BASELINE
    ))
    for pwm in mean_pwm
], dtype=np.float32)

safe_baselines = np.where(
    local_baselines > 0, local_baselines, 1.0
)

severity_ratio = reconstruction_mse / safe_baselines

severity_labels = []
recommended_actions = []

for ratio in severity_ratio:
    if ratio < MILD:
        severity_labels.append("normal")
        recommended_actions.append("No action needed.")
    elif ratio < MODERATE:
        severity_labels.append("mild")
        recommended_actions.append(
            "Log and monitor; no immediate action."
        )
    elif ratio < SEVERE:
        severity_labels.append("moderate")
        recommended_actions.append("Schedule inspection soon.")
    else:
        severity_labels.append("severe")
        recommended_actions.append(
            "Stop motor / inspect immediately."
        )


# ==========================================================
# 11. CREATE AND SAVE DETAILED CSV
# ==========================================================

time_sec = (starts + WIN / 2.0) * ROW_SEC

result = pd.DataFrame({
    "window": np.arange(len(windows)),
    "start_row": starts,
    "time_sec": time_sec,
    "mean_pwm": mean_pwm,
    "mean_irms": mean_irms,
    "mean_vrms": mean_vrms,
    "physical_motor_off": physical_motor_off_flags.astype(int),
    "pred_class_id": class_id,
    "pred_label": [LABELS[int(i)] for i in class_id],
    "class_confidence": class_confidence,
    "anomaly_probability": anomaly_probability,
    "binary_anomaly_flag": binary_flags.astype(int),
    "reconstruction_mse": reconstruction_mse,
    "reconstruction_threshold": recon_thresholds,
    "local_baseline": local_baselines,
    "severity_ratio": severity_ratio,
    "severity_label": severity_labels,
    "recommended_action": recommended_actions,
    "reconstruction_anomaly_flag": reconstruction_flags.astype(int),
    "anomaly_flag": anomaly_flags.astype(int),
    "final_decision": final_decisions
})

for i, label in enumerate(LABELS):
    result[f"probability_{label}"] = class_probability[:, i]

CSV_PATH = RESULT_DIR / f"manual_test_{stem}.csv"
result.to_csv(CSV_PATH, index=False)


# ==========================================================
# 12. PRINT MANUAL TCN TEST SUMMARY
# ==========================================================

print("\n" + "=" * 70)
print("MANUAL PURE TCN 3-HEAD TEST")
print("=" * 70)
print("File:", TEST_FILE.name)
print("Total windows:", len(result))
print("Majority class:", result["pred_label"].value_counts().idxmax())
print("Majority final decision:", result["final_decision"].value_counts().idxmax())
print("Majority severity:", result["severity_label"].value_counts().idxmax())
print("Mean class confidence:", float(class_confidence.mean()))
print("Mean anomaly probability:", float(anomaly_probability.mean()))
print("Mean reconstruction MSE:", float(reconstruction_mse.mean()))
print("Anomalous windows:", int(anomaly_flags.sum()), "/", len(result))

print("\nFinal decisions:")
print(result["final_decision"].value_counts())

print("\nSeverity tiers:")
print(result["severity_label"].value_counts())

print("\nSaved CSV:", CSV_PATH)


# ==========================================================
# 13. SAVE FIVE ORIGINAL TCN CHARTS
# ==========================================================

def save_chart(suffix):
    path = RESULT_DIR / f"manual_test_{stem}_{suffix}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.show()
    plt.close()
    print("Saved:", path)


# CHART 1 — Multiclass classification probabilities

plt.figure(figsize=(14, 7))

for i, label in enumerate(LABELS):
    plt.plot(
        time_sec, class_probability[:, i],
        label=label, linewidth=1.5
    )

plt.axhline(
    CLASS_THR,
    linestyle="--",
    label=f"Confidence threshold ({CLASS_THR:.2f})"
)

plt.xlabel("Time [s]")
plt.ylabel("Class probability")
plt.ylim(0, 1.05)
plt.title(f"TCN classification head probabilities — {stem}")
plt.legend(loc="upper right", ncol=2)
plt.grid(True, alpha=0.3)
save_chart("class_head")


# CHART 2 — Binary anomaly head

plt.figure(figsize=(14, 5))

plt.plot(
    time_sec, anomaly_probability,
    label="Anomaly probability", linewidth=2
)

plt.axhline(
    ANOM_THR,
    linestyle="--",
    label=f"Anomaly threshold ({ANOM_THR:.2f})"
)

plt.fill_between(
    time_sec, 0, 1,
    where=binary_flags,
    alpha=0.15,
    label="Anomaly detected"
)

plt.xlabel("Time [s]")
plt.ylabel("Anomaly probability")
plt.ylim(0, 1.05)
plt.title(f"TCN binary anomaly head — {stem}")
plt.legend()
plt.grid(True, alpha=0.3)
save_chart("anomaly_head")


# CHART 3 — TCN autoencoder reconstruction MSE

plt.figure(figsize=(14, 5))

plt.plot(
    time_sec, reconstruction_mse,
    label="Reconstruction MSE", linewidth=2
)

plt.plot(
    time_sec, recon_thresholds,
    linestyle="--", linewidth=2,
    label="State/PWM/load-conditioned threshold"
)

plt.fill_between(
    time_sec, 0, reconstruction_mse,
    where=reconstruction_flags,
    alpha=0.15,
    label="Reconstruction anomaly"
)

plt.xlabel("Time [s]")
plt.ylabel("Reconstruction MSE")
plt.title(f"TCN autoencoder reconstruction head — {stem}")
plt.legend()
plt.grid(True, alpha=0.3)
save_chart("reconstruction_head")


# CHART 4 — Predicted class timeline

plt.figure(figsize=(14, 5))

plt.step(
    time_sec, class_id,
    where="mid", linewidth=2
)

plt.yticks(np.arange(len(LABELS)), LABELS)
plt.xlabel("Time [s]")
plt.ylabel("Predicted class")
plt.title(f"TCN classification result timeline — {stem}")
plt.grid(True, alpha=0.3)
save_chart("class_timeline")


# CHART 5 — Reconstruction severity ratio

plt.figure(figsize=(14, 5))

plt.plot(
    time_sec, severity_ratio,
    linewidth=2, color="darkorange",
    label="Severity ratio (MSE / local baseline)"
)

for level, color in [
    (MILD, "gold"),
    (MODERATE, "orange"),
    (SEVERE, "red")
]:
    plt.axhline(
        level, linestyle="--",
        color=color, alpha=0.7
    )

plt.xlabel("Time [s]")
plt.ylabel("Severity ratio")
plt.title(f"TCN reconstruction severity — {stem}")
plt.legend()
plt.grid(True, alpha=0.3)
save_chart("severity")


# ==========================================================
# 14. COMPLETE
# ==========================================================

manual_result = result

print("\n" + "=" * 70)
print("TCN MANUAL TEST COMPLETED SUCCESSFULLY")
print("=" * 70)
print("Model:", MODEL_PATH.name)
print("Test file:", TEST_FILE.name)
print("CSV:", CSV_PATH)
print("Five charts saved in:", RESULT_DIR)
print("No training or recalibration performed.")